# 지역난방 열수요 예측: 완전한 고도화된 스태킹 앙상블

## 모델링 전략
- **구성**: 3개 규모 그룹 × 2개 계절 = 6개 모델
- **스태킹**: Prophet + CatBoost + LSTM + Ridge 메타모델
- **최적화**: 모든 모델에 Optuna + 3-Fold CV
- **총 모델**: 18개 + 6개 메타모델 = 24개
- **재현성**: 완전한 시드 고정

In [1]:
# Google Colab 환경 확인 및 패키지 설치
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Google Colab 환경에서 실행 중...")
    !pip install catboost prophet torch optuna statsmodels holidays pmdarima scikit-learn==1.3.0 --quiet
    from google.colab import files, drive
    print("패키지 설치 완료!")
else:
    print("로컬 환경에서 실행 중...")

로컬 환경에서 실행 중...


In [2]:
# 완전한 재현성을 위한 시드 고정
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 라이브러리 import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm
import pickle
import holidays
import json
import os
import copy

In [ ]:
import os
import warnings
import logging

# Prophet/Stan 로그 완전 차단
os.environ['CMDSTAN_LOGGER'] = 'ERROR'
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.ERROR)

# 특정 로거들 비활성화
for logger_name in ['prophet', 'cmdstanpy', 'pystan']:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)
    logging.getLogger(logger_name).disabled = True

In [3]:
# 머신러닝 라이브러리
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, KFold
import catboost as cb
from catboost import CatBoostRegressor

# PyTorch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# Prophet
try:
    from prophet import Prophet

except ImportError:
    print("Prophet 설치 필요")
    Prophet = None

# Optuna
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ARIMA
try:
    from pmdarima import auto_arima
    from statsmodels.tsa.arima.model import ARIMA
except ImportError:
    print("pmdarima 설치 필요")
    auto_arima = None

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"디바이스: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()

plt.rcParams['figure.figsize'] = (12, 6)
print("라이브러리 로드 완료! (시드 고정으로 재현성 보장)")

디바이스: cpu
라이브러리 로드 완료! (시드 고정으로 재현성 보장)


In [22]:
!pip install prophet --upgrade


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Huber Loss 함수 정의
def huber_loss(y_true, y_pred, delta=1.0):
    """
    Huber Loss 계산
    delta보다 작은 오차에는 제곱 손실, 큰 오차에는 선형 손실 적용
    """
    residual = np.abs(y_true - y_pred)
    condition = residual <= delta
    squared_loss = 0.5 * residual**2
    linear_loss = delta * residual - 0.5 * delta**2
    return np.where(condition, squared_loss, linear_loss).mean()
def evaluate_predictions(y_true, y_pred, delta=1.0):
    """RMSE와 Huber Loss를 동시에 계산"""
    # RMSE 계산
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Huber Loss 계산
    residual = np.abs(y_true - y_pred)
    condition = residual <= delta
    squared_loss = 0.5 * residual**2
    linear_loss = delta * residual - 0.5 * delta**2
    huber = np.where(condition, squared_loss, linear_loss).mean()
    
    return {'rmse': rmse, 'huber': huber}

def huber_score(y_true, y_pred, delta=1.0):
    """최적화용 Huber Score (작을수록 좋음)"""
    return evaluate_predictions(y_true, y_pred, delta)['huber']

print("Huber Loss 함수 정의 완료")

Huber Loss 함수 정의 완료


In [6]:
# 데이터 파일 로드
if IN_COLAB:
    print("파일 업로드 방법 선택:")
    print("1. 직접 업로드")
    print("2. Google Drive")

    method = input("선택 (1 또는 2): ")

    if method == "1":
        uploaded = files.upload()
        files_list = list(uploaded.keys())
        train_path = [f for f in files_list if 'train' in f.lower()][0]
        test_path = [f for f in files_list if 'test' in f.lower()][0]
    else:
        drive.mount('/content/drive')
        train_path = "/content/drive/MyDrive/train_heat.csv"
        test_path = "/content/drive/MyDrive/test_heat.csv"
else:
    train_path = 'train_heat.csv'
    test_path = 'test_heat.csv'

print(f"파일 경로 설정 완료")

파일 경로 설정 완료


## 1. 고도화된 데이터 (결측치 플래그 생성)

In [7]:
def load_and_preprocess_advanced(train_path, test_path):
    print("고도화된 데이터 로드 및 전처리...")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    def process_df_advanced(df):
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
        df.columns = [col.replace('train_heat.', '') for col in df.columns]

        if df['tm'].dtype == 'object':
            df['tm'] = pd.to_datetime(df['tm'])
        else:
            df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')
        
        df['year'] = df['tm'].dt.year
        df['month'] = df['tm'].dt.month
        df['day'] = df['tm'].dt.day
        df['hour'] = df['tm'].dt.hour
        df['dayofweek'] = df['tm'].dt.dayofweek
        df['dayofyear'] = df['tm'].dt.dayofyear

        # ✅ wd (풍향) 제외, 사용할 컬럼만 포함
        missing_cols = ['ta', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand'] # 열수요도 결측치 있음

        # ✅ 1단계: 결측치 플래그 생성 (NaN 변환 전에)
        print("   결측치 플래그 생성 중...")
        for col in missing_cols:
            if col in df.columns:
                # -99를 결측치로 인식하여 플래그 생성
                missing_mask = (df[col] == -99)
                df[f'{col}_missing'] = missing_mask.astype(int)
        
        # ✅ 2단계: 결측치를 NaN으로 변환
        for col in missing_cols:
            if col in df.columns:
                df[col] = df[col].replace(-99, np.nan)

        # ✅ wd 컬럼 처리: -9.9 값을 NaN으로 변환 # SVR 보간에 활용
        if 'wd' in df.columns:
            df['wd'] = df['wd'].replace(-9.9, np.nan)
            print("   wd (풍향) 컬럼의 -9.9 값을 NaN으로 변환됨")
            
        # 일사량 야간은 0 처리
        if 'si' in df.columns:
            night_mask = (df['hour'] < 8) | (df['hour'] > 18)
            df.loc[night_mask & df['si'].isna(), 'si'] = 0

        df = df.sort_values(['branch_id', 'tm'])
        
        # ✅ 3단계: 생성된 결측치 플래그 확인
        missing_flag_cols = [col for col in df.columns if col.endswith('_missing')]
        print(f"   생성된 결측치 플래그: {missing_flag_cols}")
        for col in missing_flag_cols:
            missing_count = df[col].sum()
            print(f"     {col}: {missing_count:,}개 결측치")

        return df

    train_df = process_df_advanced(train_df)
    test_df = process_df_advanced(test_df)

    print(f"   훈련: {train_df.shape}, 테스트: {test_df.shape}")
    print(f"   기간: {train_df['tm'].min()} ~ {test_df['tm'].max()}")

    return train_df, test_df

## 2-1. Feature 생성 : 시즌별 이상치 플래그 생성 (도메인 특화) _ 온도, 풍속, 강수량

In [8]:
def create_weather_outlier_flags(train_df, test_df):
    """시즌별 기상데이터 기반 이상치 플래그 (TRAIN 기준 적용)"""
    print("시즌별 기상 이상치 플래그 생성 중 (TRAIN 기준)...")
    
    # 1단계: TRAIN 데이터에서 시즌별, 지사별 임계값 계산
    outlier_thresholds = {}
    
    for branch in train_df['branch_id'].unique():
        branch_data = train_df[train_df['branch_id'] == branch]
        outlier_thresholds[branch] = {}
        
        # 시즌별로 구분하여 임계값 계산
        for season in [0, 1]:  # 0: 비난방철, 1: 난방철
            season_data = branch_data[branch_data['heating_season'] == season]
            
            if len(season_data) > 10:  # 최소 데이터 요구량
                outlier_thresholds[branch][season] = {
                    # 🌡️ 온도: 하위 10% (극한 추위)
                    'ta_q10': season_data['ta'].quantile(0.10),
                    # 💨 풍속: 상위 10% (강풍)
                    'ws_q90': season_data['ws'].quantile(0.90),
                    # 🌧️ 일강수량: 상위 10% (폭우)
                    'rn_day_q90': season_data['rn_day'].quantile(0.90)
                }
                print(f"   지사 {branch}, {'난방철' if season else '비난방철'}: 임계값 계산 완료")
    
    # 2단계: 임계값을 TRAIN과 TEST에 적용
    def apply_weather_thresholds(df, thresholds):
        df = df.copy()
        # 기본값으로 초기화
        df['cold_extreme'] = 0      # 극한 추위 (하위 10%)
        df['strong_wind'] = 0       # 강풍 (상위 10%)
        df['heavy_rain'] = 0        # 폭우 (상위 10%)
        
        for branch in df['branch_id'].unique():
            if branch in thresholds:
                branch_mask = df['branch_id'] == branch
                
                # 시즌별로 다른 임계값 적용
                for season in [0, 1]:  # 0: 비난방철, 1: 난방철
                    if season in thresholds[branch]:
                        season_mask = branch_mask & (df['heating_season'] == season)
                        season_thresholds = thresholds[branch][season]
                        
                        # 온도 이상치 (낮은 온도)
                        df.loc[season_mask, 'cold_extreme'] = (
                            df.loc[season_mask, 'ta'] < season_thresholds['ta_q10']
                        ).astype(int)
                        
                        # 풍속 이상치 (높은 풍속)
                        df.loc[season_mask, 'strong_wind'] = (
                            df.loc[season_mask, 'ws'] > season_thresholds['ws_q90']
                        ).astype(int)
                        
                        # 강수량 이상치 (많은 비)
                        df.loc[season_mask, 'heavy_rain'] = (
                            df.loc[season_mask, 'rn_day'] > season_thresholds['rn_day_q90']
                        ).astype(int)
                        
        return df
    
    # TRAIN 적용
    train_result = apply_weather_thresholds(train_df, outlier_thresholds)
    
    # TEST 적용
    test_result = apply_weather_thresholds(test_df, outlier_thresholds)
    
    print(f"   기상 이상치 플래그 생성 완료: {len(outlier_thresholds)}개 지사")
    
    return train_result, test_result, outlier_thresholds

## 2-2. Feature 생성 : 시즌 고도화된 특성

In [9]:
def create_advanced_features(df, season_type="heating"):
    df = df.copy()
    print(f"{season_type} 시즌 고도화된 특성 생성 중...")
    
    # 범주형 시간 변수 (문자열로 명시적 변환)
    df['hour_cat'] = df['hour'].astype(str)
    df['month_cat'] = df['month'].astype(str)
    df['weekday_name'] = df['dayofweek'].map(
        lambda x: ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][x]
    ).astype(str)
    
    # 순환 인코딩
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # 시즌별 월 순환 (sin, cos 포함)
    if season_type == "heating":
        heating_months = {10:0, 11:1, 12:2, 1:3, 2:4, 3:5, 4:6}
        df['heating_month_order'] = df['month'].map(heating_months)
        df['heating_month_sin'] = np.sin(2 * np.pi * df['heating_month_order'] / 7)
        df['heating_month_cos'] = np.cos(2 * np.pi * df['heating_month_order'] / 7)
    else:
        non_heating_months = {5:0, 6:1, 7:2, 8:3, 9:4}
        df['non_heating_month_order'] = df['month'].map(non_heating_months)
        df['non_heating_month_sin'] = np.sin(2 * np.pi * df['non_heating_month_order'] / 5)
        df['non_heating_month_cos'] = np.cos(2 * np.pi * df['non_heating_month_order'] / 5)
    
    # 브랜치 ID (문자열로 변환)
    df['branch_id'] = df['branch_id'].astype(str)
    
    # 고급 기상 범주형 변수
    df['temp_category'] = 'Normal'
    df.loc[df['ta'] < -10, 'temp_category'] = 'VeryCold'
    df.loc[(df['ta'] >= -10) & (df['ta'] < 0), 'temp_category'] = 'Cold'
    df.loc[(df['ta'] >= 0) & (df['ta'] < 10), 'temp_category'] = 'Cool'
    df.loc[(df['ta'] >= 10) & (df['ta'] < 25), 'temp_category'] = 'Normal'
    df.loc[df['ta'] >= 25, 'temp_category'] = 'Hot'
    df['temp_category'] = df['temp_category'].astype(str)
    
    if season_type == "heating":
        df['cold_warning_level'] = 'Normal'
        df.loc[df['ta'] <= -12, 'cold_warning_level'] = 'ColdAdvisory'
        df.loc[df['ta'] <= -15, 'cold_warning_level'] = 'ColdWarning'
        df['cold_warning_level'] = df['cold_warning_level'].astype(str)
    
    df['wind_category'] = 'Weak'
    df.loc[df['ws'] >= 5.0, 'wind_category'] = 'Moderate'
    df.loc[df['ws'] >= 10.0, 'wind_category'] = 'Strong'
    df['wind_category'] = df['wind_category'].astype(str)
    
    # 공휴일/피크시간
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: x in kr_holidays)
    df['holiday_type'] = df['is_holiday'].map({False: 'Weekday', True: 'Holiday'}).astype(str)
    
    df['peak_time'] = 'Normal'
    df.loc[(df['hour'] >= 0) & (df['hour'] <= 6), 'peak_time'] = 'Dawn'
    df.loc[(df['hour'] > 6) & (df['hour'] <= 11), 'peak_time'] = 'Morning'
    df.loc[(df['hour'] > 11) & (df['hour'] <= 18), 'peak_time'] = 'Afternoon'
    df.loc[(df['hour'] > 18) & (df['hour'] <= 23), 'peak_time'] = 'Evening'
    df['peak_time'] = df['peak_time'].astype(str)
    
    # 고급 수치형 특성
    df['HDD18'] = np.maximum(0, 18 - df['ta'])
    # df['HDD20'] = np.maximum(0, 20 - df['ta'])
    
    # apparent_temp는 난방시즌에만 생성
    if season_type == "heating":
        def calculate_apparent_temp(ta, ws):
            winter_at = 13.12 + 0.6215 * ta - 11.37 * (ws * 3.6)**0.16 + 0.3965 * ta * (ws * 3.6)**0.16
            return winter_at
        
        df['apparent_temp'] = calculate_apparent_temp(df['ta'], df['ws'])
        df['apparent_temp'] = df['apparent_temp'].fillna(0)
    
    for lag in [3, 6, 24]:
        df[f'ta_lag_{lag}h'] = df.groupby('branch_id')['ta'].shift(lag).fillna(0) # 초기 비어있는 값은 0으로 반영
    
    for window in [6, 12, 24]:
        df[f'ta_ma_{window}h'] = df.groupby('branch_id')['ta'].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
    
    df['ta_diff_3h'] = df.groupby('branch_id')['ta'].diff(3)
    df['ta_diff_6h'] = df.groupby('branch_id')['ta'].diff(6)
    # diff 변수 결측치를 0으로 채우기
    df['ta_diff_3h'] = df['ta_diff_3h'].fillna(0)
    df['ta_diff_6h'] = df['ta_diff_6h'].fillna(0)
    
    daily_stats = df.groupby(['branch_id', df['tm'].dt.date]).agg({
        'ta': ['min', 'max', 'mean']
    }).round(2)
    daily_stats.columns = ['daily_ta_min', 'daily_ta_max', 'daily_ta_mean']
    daily_stats['daily_temp_range'] = daily_stats['daily_ta_max'] - daily_stats['daily_ta_min']
    
    df = df.merge(
        daily_stats.reset_index(),
        left_on=['branch_id', df['tm'].dt.date],
        right_on=['branch_id', 'tm'],
        how='left',
        suffixes=('', '_daily')
    )
    
    print(f"   {season_type} 시즌 고도화된 특성 생성 완료: {df.shape[1]}개 컬럼")
    return df

## 2-3. 결측치 보간 (Branch별 SVR)

In [10]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

def apply_svr_interpolation(df, target_col='heat_demand', is_train=True):
    """SVR 기반 고급 보간 함수 (폴백 없음, 실패시 에러)"""
    print(f"SVR 보간 적용 중 ({'TRAIN' if is_train else 'TEST'} 데이터)...")
    
    df_interpolated = df.copy()
    
    # 보간 대상 컬럼 확장
    interpolation_cols = ['ta', 'hm', 'ws', 'wd', 'rn_day', 'rn_hr1', 'si', 'ta_chi', 'heat_demand']
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    available_cols = [col for col in interpolation_cols if col in numeric_cols]
    
    print(f"   보간 대상 컬럼: {available_cols}")
    
    # 시간 특성 생성 (SVR용)
    df_interpolated['hour'] = df_interpolated['tm'].dt.hour
    df_interpolated['day_of_year'] = df_interpolated['tm'].dt.dayofyear
    df_interpolated['month'] = df_interpolated['tm'].dt.month
    df_interpolated['dayofweek'] = df_interpolated['tm'].dt.dayofweek
    
    # 브랜치별 SVR 보간
    for branch in tqdm(df['branch_id'].unique(), desc="지사별 SVR 보간"):
        branch_mask = df_interpolated['branch_id'] == branch
        branch_data = df_interpolated[branch_mask].copy().sort_values('tm')
        
        if len(branch_data) < 24:  # 최소 데이터 요구량
            raise ValueError(f"지사 {branch}: 데이터 부족 ({len(branch_data)}개). 최소 24개 필요.")
            
        for col in available_cols:
            if col not in branch_data.columns:
                continue
                
            missing_mask = branch_data[col].isna()
            missing_count = missing_mask.sum()
            total_count = len(branch_data)
            
            if missing_count == 0:  # 결측치가 없으면 스킵
                continue
            
            try:
                # 훈련용 데이터 (결측이 아닌 것들)
                train_mask = ~missing_mask
                
                if train_mask.sum() < 10:  # 최소 10개 이상의 훈련 데이터 필요
                    raise ValueError(f"지사 {branch}, 컬럼 {col}: 훈련 데이터 부족 ({train_mask.sum()}개). 최소 10개 필요.")
                
                # 특성: 시간, 연중일, 월, 요일
                feature_cols = ['hour', 'day_of_year', 'month', 'dayofweek']
                X_train = branch_data.loc[train_mask, feature_cols].values
                y_train = branch_data.loc[train_mask, col].values
                
                # 예측할 데이터
                X_pred = branch_data.loc[missing_mask, feature_cols].values
                
                if len(X_pred) == 0:
                    continue
                
                # 스케일링
                scaler_X = StandardScaler()
                scaler_y = StandardScaler()
                
                X_train_scaled = scaler_X.fit_transform(X_train)
                y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
                
                # SVR 모델 훈련
                svr = SVR(kernel='rbf', C=1.0, gamma='scale', epsilon=0.1)
                svr.fit(X_train_scaled, y_train_scaled)
                
                # 예측
                X_pred_scaled = scaler_X.transform(X_pred)
                y_pred_scaled = svr.predict(X_pred_scaled)
                y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
                
                # 결과 할당
                df_interpolated.loc[branch_mask & missing_mask, col] = y_pred
                
                print(f"     지사 {branch}, {col}: {missing_count}개 결측치 SVR 보간 완료")
                
            except Exception as e:
                print(f"❌ 지사 {branch}, 컬럼 {col} SVR 보간 실패:")
                print(f"   에러: {str(e)}")
                print(f"   결측치: {missing_count}/{total_count}개")
                print(f"   훈련 데이터: {train_mask.sum() if 'train_mask' in locals() else 'N/A'}개")
                raise e  # ✅ 폴백 없음, 에러 발생시키고 중단
    
    # 최종 결측치 확인 및 처리
    for col in available_cols:
        if col in df_interpolated.columns:
            remaining_nan = df_interpolated[col].isna().sum()
            if remaining_nan > 0:
                print(f"⚠️ {col}: SVR 보간 후에도 {remaining_nan}개 결측치 남음")
                # Forward/Backward fill로 최종 처리
                df_interpolated[col] = df_interpolated[col].ffill().bfill()
                
                # 여전히 결측치가 있으면 에러
                final_nan = df_interpolated[col].isna().sum()
                if final_nan > 0:
                    raise ValueError(f"{col}: 모든 보간 방법 실패, {final_nan}개 결측치 남음")
    
    print(f"   ✅ SVR 보간 완료")
    return df_interpolated


## 3-1. 규모별 그룹 분할 및 CV 설정


In [11]:
# 1. heating_season 컬럼 추가 함수
def add_heating_season(df):
    """난방 시즌 컬럼 추가"""
    df = df.copy()
    df['heating_season'] = 0  # 기본값: 비난방
    heating_months = [10, 11, 12, 1, 2, 3, 4]  # 10월~4월: 난방시즌
    df.loc[df['month'].isin(heating_months), 'heating_season'] = 1
    return df

# 2. 시즌별 데이터 분할 함수
def split_by_season_only(df):
    """시즌별로만 분할 (2개 그룹)"""
    groups = {}
    
    for season in [0, 1]:  # 0: 비난방, 1: 난방
        season_name = 'heating' if season == 1 else 'non_heating'
        season_data = df[df['heating_season'] == season].copy()
        groups[season_name] = season_data
                
    return groups

# 3. 연도 기반 CV 분할 함수
def create_year_based_cv_splits(df, group_name=""):
    """연도 기반 3-Fold CV 분할 생성"""
    print(f"{group_name} 그룹 - 연도 기반 3-Fold CV 분할 생성...")
    
    # 연도별 데이터 분포 확인
    year_counts = df['year'].value_counts().sort_index()
    print(f"   연도별 데이터 분포:")
    for year, count in year_counts.items():
        print(f"     {year}년: {count:,}개")
    
    # 3-Fold CV: 2021, 2022, 2023년 각각 validation으로 사용
    cv_splits = []
    
    for val_year in [2021, 2022, 2023]:
        train_mask = df['year'] != val_year
        val_mask = df['year'] == val_year
        
        train_indices = df[train_mask].index.tolist()
        val_indices = df[val_mask].index.tolist()
        
        cv_splits.append((train_indices, val_indices))
        
        print(f"   Fold {val_year}: 훈련 {len(train_indices):,}개, 검증 {len(val_indices):,}개")
    
    return cv_splits

## 3-2. 데이터셋 전처리 및 특성 생성

In [12]:
# 1. 기본 전처리 (공통)
print("1️⃣ 기본 전처리...")
train_df, test_df = load_and_preprocess_advanced(train_path, test_path)

# 2. heating_season 컬럼 추가
print("2️⃣ heating_season 컬럼 추가...")
train_df = add_heating_season(train_df)
test_df = add_heating_season(test_df)

# 3. ✅ 결측치 보간 먼저! (파생변수 생성 전)
print("3️⃣ 결측치 SVR 보간 적용...")
print("\n🔧 훈련 데이터 SVR 보간:")
train_df = apply_svr_interpolation(train_df, 'heat_demand', is_train=True)

print("\n🔧 테스트 데이터 SVR 보간:")
test_df = apply_svr_interpolation(test_df, is_train=False)

# 4. 이상치 플래그 생성 (보간 후)
print("4️⃣ 이상치 플래그 생성...")
train_df, test_df, weather_thresholds = create_weather_outlier_flags(train_df, test_df)

# 5. 그룹별로 먼저 분할
print("5️⃣ 시즌별 그룹 분할...")
train_groups = split_by_season_only(train_df)
test_groups = split_by_season_only(test_df)

# 6. 각 그룹별로 시즌에 맞는 특성 생성 (보간 완료된 데이터로)
print("6️⃣ 그룹별 고도화된 특성 생성...")

# heating 그룹 특성 생성
print("  🔥 heating 그룹 특성 생성 중...")
train_groups['heating'] = create_advanced_features(train_groups['heating'], "heating")
test_groups['heating'] = create_advanced_features(test_groups['heating'], "heating")

# non_heating 그룹 특성 생성  
print("  ❄️ non_heating 그룹 특성 생성 중...")
train_groups['non_heating'] = create_advanced_features(train_groups['non_heating'], "non_heating")
test_groups['non_heating'] = create_advanced_features(test_groups['non_heating'], "non_heating")

print(f"\n📊 그룹별 처리 후 데이터 크기:")
for group_name in ['heating', 'non_heating']:
    print(f"   {group_name:12s}: 훈련 {train_groups[group_name].shape}, 테스트 {test_groups[group_name].shape}")

print(f"\n✅ 모든 전처리 완료!")

# 7. 그룹별 CV 분할 생성 및 결과 확인
print("7️⃣ 그룹별 CV 분할 미리보기:")
for group_name, group_data in train_groups.items():
    if len(group_data) > 100:
        cv_splits = create_year_based_cv_splits(group_data, group_name)
        print(f"   {group_name}: {len(cv_splits)}개 fold 생성됨")
        
        # 결측치 최종 확인
        weather_cols = ['ta', 'hm', 'ws', 'rn_day', 'rn_hr1', 'si', 'ta_chi', 'apparent_temp']
        total_missing = 0
        for col in weather_cols:
            if col in group_data.columns:
                missing = group_data[col].isna().sum()
                total_missing += missing
                if missing > 0:
                    print(f"     ⚠️ {col}: {missing}개 결측치 남음")
        
        if total_missing == 0:
            print(f"     ✅ {group_name}: 모든 기상변수 결측치 해결됨")
        print()

1️⃣ 기본 전처리...
고도화된 데이터 로드 및 전처리...
   결측치 플래그 생성 중...
   wd (풍향) 컬럼 삭제됨
   생성된 결측치 플래그: ['ta_missing', 'ws_missing', 'rn_day_missing', 'rn_hr1_missing', 'hm_missing', 'si_missing', 'ta_chi_missing']
     ta_missing: 12,997개 결측치
     ws_missing: 18,815개 결측치
     rn_day_missing: 18,626개 결측치
     rn_hr1_missing: 19,154개 결측치
     hm_missing: 39,717개 결측치
     si_missing: 232,922개 결측치
     ta_chi_missing: 20개 결측치
   결측치 플래그 생성 중...
   wd (풍향) 컬럼 삭제됨
   생성된 결측치 플래그: ['ta_missing', 'ws_missing', 'rn_day_missing', 'rn_hr1_missing', 'hm_missing', 'si_missing', 'ta_chi_missing']
     ta_missing: 2,355개 결측치
     ws_missing: 4,806개 결측치
     rn_day_missing: 4,411개 결측치
     rn_hr1_missing: 4,590개 결측치
     hm_missing: 5,875개 결측치
     si_missing: 75,754개 결측치
     ta_chi_missing: 1개 결측치
   훈련: (499301, 23), 테스트: (166915, 23)
   기간: 2021-01-01 01:00:00 ~ 2025-01-01 00:00:00
2️⃣ heating_season 컬럼 추가...
3️⃣ 결측치 SVR 보간 적용...

🔧 훈련 데이터 SVR 보간:
SVR 보간 적용 중 (TRAIN 데이터)...
   보간 대상 컬럼: ['ta', 'hm', 'ws', 'rn_da

지사별 SVR 보간:   0%|          | 0/19 [00:00<?, ?it/s]

     지사 A, ta: 4개 결측치 SVR 보간 완료
     지사 A, hm: 4개 결측치 SVR 보간 완료
     지사 A, ws: 18개 결측치 SVR 보간 완료
     지사 A, rn_day: 277개 결측치 SVR 보간 완료
     지사 A, rn_hr1: 281개 결측치 SVR 보간 완료
     지사 A, si: 16개 결측치 SVR 보간 완료
     지사 B, ta: 212개 결측치 SVR 보간 완료
     지사 B, hm: 2697개 결측치 SVR 보간 완료
     지사 B, ws: 433개 결측치 SVR 보간 완료
     지사 B, rn_day: 309개 결측치 SVR 보간 완료
     지사 B, rn_hr1: 332개 결측치 SVR 보간 완료
     지사 B, si: 16개 결측치 SVR 보간 완료
     지사 C, ta: 23개 결측치 SVR 보간 완료
     지사 C, hm: 23개 결측치 SVR 보간 완료
     지사 C, ws: 182개 결측치 SVR 보간 완료
     지사 C, rn_day: 226개 결측치 SVR 보간 완료
     지사 C, rn_hr1: 239개 결측치 SVR 보간 완료
     지사 C, si: 16개 결측치 SVR 보간 완료
     지사 D, ta: 3739개 결측치 SVR 보간 완료
     지사 D, hm: 7698개 결측치 SVR 보간 완료
     지사 D, ws: 5680개 결측치 SVR 보간 완료
     지사 D, rn_day: 4064개 결측치 SVR 보간 완료
     지사 D, rn_hr1: 4112개 결측치 SVR 보간 완료
     지사 D, si: 17개 결측치 SVR 보간 완료
     지사 D, ta_chi: 1개 결측치 SVR 보간 완료
     지사 E, ta: 3739개 결측치 SVR 보간 완료
     지사 E, hm: 7698개 결측치 SVR 보간 완료
     지사 E, ws: 5680개 결측치 SVR 보간 완료
     지사 E, rn_da

지사별 SVR 보간:   0%|          | 0/19 [00:00<?, ?it/s]

     지사 A, ta: 8개 결측치 SVR 보간 완료
     지사 A, hm: 9개 결측치 SVR 보간 완료
     지사 A, ws: 14개 결측치 SVR 보간 완료
     지사 A, rn_day: 68개 결측치 SVR 보간 완료
     지사 A, rn_hr1: 71개 결측치 SVR 보간 완료
     지사 A, si: 17개 결측치 SVR 보간 완료
     지사 B, ta: 50개 결측치 SVR 보간 완료
     지사 B, hm: 3339개 결측치 SVR 보간 완료
     지사 B, ws: 352개 결측치 SVR 보간 완료
     지사 B, rn_day: 52개 결측치 SVR 보간 완료
     지사 B, rn_hr1: 54개 결측치 SVR 보간 완료
     지사 B, si: 17개 결측치 SVR 보간 완료
     지사 C, ta: 17개 결측치 SVR 보간 완료
     지사 C, hm: 18개 결측치 SVR 보간 완료
     지사 C, ws: 70개 결측치 SVR 보간 완료
     지사 C, rn_day: 71개 결측치 SVR 보간 완료
     지사 C, rn_hr1: 77개 결측치 SVR 보간 완료
     지사 C, si: 17개 결측치 SVR 보간 완료
     지사 D, ta: 716개 결측치 SVR 보간 완료
     지사 D, hm: 717개 결측치 SVR 보간 완료
     지사 D, ws: 904개 결측치 SVR 보간 완료
     지사 D, rn_day: 904개 결측치 SVR 보간 완료
     지사 D, rn_hr1: 916개 결측치 SVR 보간 완료
     지사 E, ta: 716개 결측치 SVR 보간 완료
     지사 E, hm: 717개 결측치 SVR 보간 완료
     지사 E, ws: 904개 결측치 SVR 보간 완료
     지사 E, rn_day: 904개 결측치 SVR 보간 완료
     지사 E, rn_hr1: 916개 결측치 SVR 보간 완료
     지사 F, ta: 91개 결측치 SV

In [13]:
print("7️⃣ 그룹별 CV 분할 미리보기:")
for group_name, group_data in train_groups.items():
    if len(group_data) > 100:
        cv_splits = create_year_based_cv_splits(group_data, group_name)
        print(f"   {group_name}: {len(cv_splits)}개 fold 생성됨")
        
        # 결측치 최종 확인
        weather_cols = [# 기본 시간 변수 추가
            'hour', 'month', 'day',
            # 기상 변수
            'ta', 'hm', 'ws', 'rn_day', 'rn_hr1', 'si', 'ta_chi',  # rn_hr1, ta_chi 추가
            # 파생 변수
            'HDD18', 'apparent_temp',  # HDD20 제거
            # 순환 인코딩
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 
            'dayofweek_sin', 'dayofweek_cos',
            # 시계열 특성
            'ta_lag_3h', 'ta_lag_6h', 'ta_lag_24h', 
            'ta_ma_6h', 'ta_ma_12h', 'ta_ma_24h',
            'ta_diff_3h', 'ta_diff_6h', 
            # 일별 통계
            'daily_ta_min', 'daily_ta_max', 'daily_ta_mean', 'daily_temp_range']
        total_missing = 0
        for col in weather_cols:
            if col in group_data.columns:
                missing = group_data[col].isna().sum()
                total_missing += missing
                if missing > 0:
                    print(f"     ⚠️ {col}: {missing}개 결측치 남음")
        
        if total_missing == 0:
            print(f"     ✅ {group_name}: 모든 기상변수 결측치 해결됨")
        print()

7️⃣ 그룹별 CV 분할 미리보기:
non_heating 그룹 - 연도 기반 3-Fold CV 분할 생성...
   연도별 데이터 분포:
     2021년: 69,768개
     2022년: 69,768개
     2023년: 69,768개
   Fold 2021: 훈련 139,536개, 검증 69,768개
   Fold 2022: 훈련 139,536개, 검증 69,768개
   Fold 2023: 훈련 139,536개, 검증 69,768개
   non_heating: 3개 fold 생성됨
     ⚠️ ta_lag_3h: 3개 결측치 남음
     ⚠️ ta_lag_6h: 6개 결측치 남음
     ⚠️ ta_lag_24h: 24개 결측치 남음

heating 그룹 - 연도 기반 3-Fold CV 분할 생성...
   연도별 데이터 분포:
     2021년: 96,653개
     2022년: 96,672개
     2023년: 96,672개
   Fold 2021: 훈련 193,344개, 검증 96,653개
   Fold 2022: 훈련 193,325개, 검증 96,672개
   Fold 2023: 훈련 193,325개, 검증 96,672개
   heating: 3개 fold 생성됨
     ⚠️ apparent_temp: 78개 결측치 남음
     ⚠️ ta_lag_3h: 3개 결측치 남음
     ⚠️ ta_lag_6h: 6개 결측치 남음
     ⚠️ ta_lag_24h: 24개 결측치 남음



## 4. 모델별 피쳐 정의

Prophet

- 시간 변수와 기본 기상 변수 중심
- 너무 많은 변수보다는 핵심 특성에 집중

CatBoost

- 범주형 변수와 플래그 변수를 최대한 활용
- 결측치/이상치 플래그로 데이터 품질 정보 제공

LSTM

- 수치형 변수 중심으로 단순화
- 시퀀스 학습에 집중할 수 있도록 노이즈 최소화

In [14]:
def define_model_features():
    # Prophet은 자체 시계열 분해 능력이 있음
    prophet_features = {
        'basic': [
            'ta', 'hm', 'ws', 'HDD18', 'apparent_temp'
        ],
        'seasonal': [
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 
            'dayofweek_sin', 'dayofweek_cos',
            'ta_lag_3h', 'ta_lag_6h'  # 짧은 lag만 (Prophet 자체 시계열 처리 보완) # ma, diff는 불필요 (Prophet이 자체 처리)
        ]
    }
    # CatBoost: 모든 특성 활용 (범주형 + 시계열 + 플래그)
    catboost_features = {
        'numerical': [
            'day', 'dayofyear',
            # 기상 변수
            'ta', 'hm', 'ws', 'rn_day', 'rn_hr1', 'si', 'ta_chi',
            # 파생 변수
            'HDD18',
            # 순환 인코딩
            'hour_sin', 'hour_cos', #'month_sin', 'month_cos' 제외 (중복)
            'dayofweek_sin', 'dayofweek_cos',
            # 모든 시계열 특성 (CatBoost는 직접 학습)
            'ta_lag_3h', 'ta_lag_6h', 'ta_lag_24h', 
            'ta_ma_6h', 'ta_ma_12h', 'ta_ma_24h',
            'ta_diff_3h', 'ta_diff_6h', 
            # 일별 통계
            'daily_ta_min', 'daily_ta_max', 'daily_ta_mean', 'daily_temp_range'
        ],
        'categorical': [
            'branch_id', 'hour_cat', 'month_cat', 'weekday_name', 
            'temp_category', 'wind_category', 'holiday_type', 'peak_time'
        ],
        'flags': [
            # 결측치 플래그 (모든 기상 변수)
            'ta_missing', 'ws_missing', 'rn_day_missing', 
            'rn_hr1_missing', 'hm_missing', 'si_missing', 'ta_chi_missing',
            # 이상치 플래그 
            'cold_extreme', 'strong_wind', 'heavy_rain'
        ]
    }
     # LSTM: 핵심 특성 + 전처리된 시계열 (lag 제외)
    lstm_features = {
        'numerical': [
            # 기본 시간 변수 추가
            'day', 'dayofyear',
            # 핵심 기상 변수
            'ta', 'hm', 'ws', 'rn_day', 'si',
            # 파생 변수
            'HDD18',
            # 순환 인코딩
            'hour_sin', 'hour_cos', #'month_sin', 'month_cos' 제외 (중복)
            'dayofweek_sin', 'dayofweek_cos',
            # 핵심 이동 평균만
            'ta_ma_6h', 'ta_ma_12h', 'ta_ma_24h',
            # 일별 통계 (핵심만)
            'daily_ta_mean', 'daily_temp_range' # lag, diff 제거 - LSTM sequence로 대체
        ], 
        'categorical_encoded': ['branch_id']
    }
    
    # heating 시즌별 특성 추가
    prophet_heating = copy.deepcopy(prophet_features)
    prophet_heating['basic'].append('apparent_temp')
    prophet_heating['seasonal'].extend(['heating_month_order'])
     # prophet_non_heating도 non_heating_month_order 추가
    prophet_non_heating = copy.deepcopy(prophet_features)
    prophet_non_heating['seasonal'].append('non_heating_month_order')

    # 난방시즌용 CatBoost (한파 경보 + 시즌별 순환 추가)
    catboost_heating = copy.deepcopy(catboost_features)
    catboost_heating['categorical'].append('cold_warning_level')
    catboost_heating['seasonal'] = [
        'apparent_temp', 'heating_month_order', 
        'heating_month_sin', 'heating_month_cos'
    ]
    # 비난방시즌용 CatBoost (시즌별 순환 추가)
    catboost_non_heating = copy.deepcopy(catboost_features)
    catboost_non_heating['seasonal'] = [
        'non_heating_month_order', 'non_heating_month_sin', 'non_heating_month_cos'
    ]
    # LSTM 난방시즌용 특성 추가
    lstm_heating = copy.deepcopy(lstm_features)
    lstm_heating['numerical'].extend([
        'apparent_temp', 'heating_month_order', 'heating_month_sin', 'heating_month_cos'  # 추가
    ])
    # LSTM 비난방시즌용 특성 추가
    lstm_non_heating = copy.deepcopy(lstm_features)
    lstm_non_heating['numerical'].extend([
        'non_heating_month_order', 'non_heating_month_sin', 'non_heating_month_cos'  # 추가
    ])
    
    return {
        'prophet_heating': prophet_heating,
        'prophet_non_heating': prophet_non_heating,  # apparent_temp 없음
        'catboost_heating': catboost_heating,
        'catboost_non_heating': catboost_non_heating,
        'lstm_heating': lstm_heating,
        'lstm_non_heating': lstm_non_heating
    }

model_features = define_model_features()

print("모델별 피쳐 정의 완료:")
print("=" * 50)
for model_name, features in model_features.items():
    total_features = sum(len(v) if isinstance(v, list) else 0 for v in features.values())
    print(f"{model_name:20s}: {total_features}개 피쳐")
    for feature_type, feature_list in features.items():
        if isinstance(feature_list, list):
            print(f"  {feature_type:15s}: {len(feature_list)}개")

모델별 피쳐 정의 완료:
prophet_heating     : 17개 피쳐
  basic          : 6개
  seasonal       : 11개
prophet_non_heating : 14개 피쳐
  basic          : 5개
  seasonal       : 9개
catboost_heating    : 50개 피쳐
  numerical      : 27개
  categorical    : 9개
  flags          : 10개
  seasonal       : 4개
catboost_non_heating: 48개 피쳐
  numerical      : 27개
  categorical    : 8개
  flags          : 10개
  seasonal       : 3개
lstm_heating        : 22개 피쳐
  numerical      : 21개
  categorical_encoded: 1개
lstm_non_heating    : 21개 피쳐
  numerical      : 20개
  categorical_encoded: 1개


## 5. 모델 클래스들 (Huber Loss 지원)

In [15]:
# GPU 메모리 최적화된 LSTM 모델
class LSTMNet(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 1)
        
    def forward(self, x):
        # GPU 메모리 효율성을 위한 최적화
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]  # 마지막 시점만 사용
        del lstm_out  # 메모리 해제
        
        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        return out

# Time Series Dataset 클래스 추가
class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, sequence_length=24):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.sequence_length = sequence_length
        
    def __len__(self):
        return len(self.X) - self.sequence_length + 1
    
    def __getitem__(self, idx):
        return (
            self.X[idx:idx+self.sequence_length],
            self.y[idx+self.sequence_length-1]
        )

# Huber Loss 클래스 추가 (LSTM용)
class HuberLoss(nn.Module):
    """PyTorch용 Huber Loss"""
    def __init__(self, delta=1.0):
        super(HuberLoss, self).__init__()
        self.delta = delta

    def forward(self, y_pred, y_true):
        residual = torch.abs(y_true - y_pred)
        condition = residual <= self.delta
        squared_loss = 0.5 * residual**2
        linear_loss = self.delta * residual - 0.5 * self.delta**2
        return torch.where(condition, squared_loss, linear_loss).mean()

print("LSTM 모델 클래스, TimeSeriesDataset 및 Huber Loss 정의 완료")

LSTM 모델 클래스, TimeSeriesDataset 및 Huber Loss 정의 완료


## 6. Prophet, CatBoost, LSTM 모델 클래스 (Optuna 최적화)

In [27]:
class ProphetOptimizedModel:
    def __init__(self, season_type="heating"):
        self.models = {}
        self.best_params = {}
        self.season_type = season_type
        
        
    def optimize_hyperparameters(self, df, cv_splits, target_col='heat_demand', n_trials=30):
        """연도 기반 CV를 사용한 하이퍼파라미터 최적화"""
        print(f"Prophet Huber Loss 하이퍼파라미터 최적화 중... (trials: {n_trials})")
        
        def objective(trial):
            # 하이퍼파라미터 샘플링
            changepoint_prior_scale = trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True)
            seasonality_prior_scale = trial.suggest_float('seasonality_prior_scale', 0.1, 10, log=True)
            holidays_prior_scale = trial.suggest_float('holidays_prior_scale', 0.1, 10, log=True)
            seasonality_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
            
            cv_scores = []
            
            # 연도 기반 3-Fold CV
            for fold, (train_idx, val_idx) in enumerate(cv_splits):
                fold_predictions = []
                fold_targets = []
                
                train_fold = df.iloc[train_idx]
                val_fold = df.iloc[val_idx]
                
                # 지사별 모델 훈련 및 예측
                for branch in df['branch_id'].unique():
                    branch_train = train_fold[train_fold['branch_id'] == branch]
                    branch_val = val_fold[val_fold['branch_id'] == branch]
                    
                    if len(branch_train) < 50 or len(branch_val) == 0:
                        continue
                    
                    try:
                        # Prophet 데이터 준비 (이미 보간된 데이터 사용)
                        prophet_df = pd.DataFrame({
                            'ds': pd.to_datetime(branch_train['tm']),
                            'y': branch_train[target_col]
                        })
                        
                        # 시즌별 피쳐 설정 사용
                        feature_config = model_features[f'prophet_{self.season_type}']
                        regressors = feature_config['basic'] + feature_config['seasonal']
                        
                        for reg in regressors:
                            if reg in branch_train.columns:
                                prophet_df[reg] = branch_train[reg].values
                        
                        # Prophet 모델 생성 및 훈련
                        model = Prophet(
                            changepoint_prior_scale=changepoint_prior_scale,
                            seasonality_prior_scale=seasonality_prior_scale,
                            holidays_prior_scale=holidays_prior_scale,
                            seasonality_mode=seasonality_mode,
                            daily_seasonality=True,
                            weekly_seasonality=True,
                            yearly_seasonality=True
                        )
                        
                        # 회귀변수 추가
                        for reg in regressors:
                            if reg in prophet_df.columns and reg not in ['ds', 'y']:
                                model.add_regressor(reg)
                        
                        model.fit(prophet_df)
                        
                        # 예측 데이터 준비
                        future_df = pd.DataFrame({
                            'ds': pd.to_datetime(branch_val['tm'])
                        })
                        
                        for reg in regressors:
                            if reg in branch_val.columns:
                                future_df[reg] = branch_val[reg].values
                        
                        # 예측 실행
                        forecast = model.predict(future_df)
                        predictions = np.maximum(forecast['yhat'].values, 0)
                        
                        actual_values = branch_val[target_col].values
                        valid_mask = ~np.isnan(actual_values) & ~np.isnan(predictions)
                        
                        if valid_mask.sum() > 0:
                            fold_predictions.extend(predictions[valid_mask])
                            fold_targets.extend(actual_values[valid_mask])
                    
                    except Exception as e:
                        print(f"❌ 최적화 중 지사 {branch} Fold {fold+1} 실패:")
                        print(f"   에러: {str(e)}")
                        raise e
                
                # Fold별 Huber Loss 계산
                # 수정된 코드
                if len(fold_predictions) > 10:
                    # ✅ list를 numpy array로 변환
                    fold_targets_array = np.array(fold_targets)
                    fold_predictions_array = np.array(fold_predictions)
                    
                    huber = huber_score(fold_targets_array, fold_predictions_array, delta=1.0)
                    cv_scores.append(huber)
                    print(f"   Fold {fold+1}: Huber Loss = {huber:.4f} ({len(fold_predictions)}개 예측)")
                else:
                    print(f"   Fold {fold+1}: 유효한 예측 부족 ({len(fold_predictions)}개)")
            
            if len(cv_scores) == 0:
                raise RuntimeError("모든 Fold에서 Prophet 최적화 실패")
                
            return np.mean(cv_scores)
        
        # Optuna 최적화 실행
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_params = study.best_params
        print(f"   Prophet 최적 Huber Loss: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_params}")
        return study.best_value

    def fit(self, df, target_col='heat_demand'):
        """최적화된 파라미터로 전체 데이터 훈련"""
        print(f"Prophet 모델 훈련 중...")
        
        branches = df['branch_id'].unique()
        success_count = 0
        
        # 시즌별 피쳐 설정 사용
        feature_config = model_features[f'prophet_{self.season_type}']
        regressors = feature_config['basic'] + feature_config['seasonal']
        
        print(f"   사용할 regressors: {regressors}")

        for branch in tqdm(branches, desc="Prophet 지사별 훈련"):
            branch_data = df[df['branch_id'] == branch].copy()

            if len(branch_data) < 50:
                print(f"⚠️ 지사 {branch}: 데이터 부족 ({len(branch_data)}개) - 스킵")
                continue

            try:
                # Prophet 데이터 준비
                prophet_df = pd.DataFrame({
                    'ds': branch_data['tm'],
                    'y': branch_data[target_col]
                })

                # 회귀변수 추가
                missing_regressors = []
                for reg in regressors:
                    if reg in branch_data.columns:
                        prophet_df[reg] = branch_data[reg].values
                    else:
                        missing_regressors.append(reg)
                
                if missing_regressors:
                    print(f"⚠️ 지사 {branch}: 누락된 regressors: {missing_regressors}")

                # 최적화된 파라미터로 모델 생성
                model = Prophet(
                    changepoint_prior_scale=self.best_params.get('changepoint_prior_scale', 0.05),
                    seasonality_prior_scale=self.best_params.get('seasonality_prior_scale', 10.0),
                    holidays_prior_scale=self.best_params.get('holidays_prior_scale', 10.0),
                    seasonality_mode=self.best_params.get('seasonality_mode', 'multiplicative'),
                    daily_seasonality=True,
                    weekly_seasonality=True,
                    yearly_seasonality=True  # ✅ 연간 계절성 활성화
                )

                # 회귀변수 추가
                added_regressors = []
                for reg in regressors:
                    if reg in prophet_df.columns and reg not in ['ds', 'y']:
                        model.add_regressor(reg)
                        added_regressors.append(reg)

                # 데이터 품질 확인
                if prophet_df['y'].isna().sum() > 0:
                    print(f"⚠️ 지사 {branch}: 타겟 변수에 결측치 {prophet_df['y'].isna().sum()}개")
                
                model.fit(prophet_df)
                self.models[branch] = model
                success_count += 1

            except Exception as e:
                print(f"❌ 지사 {branch} Prophet 훈련 실패:")
                print(f"   에러: {str(e)}")
                print(f"   데이터 크기: {len(branch_data)}")
                print(f"   Prophet 데이터프레임 크기: {prophet_df.shape}")
                print(f"   추가된 regressors: {added_regressors}")
                print(f"   타겟 변수 통계:")
                print(f"     평균: {prophet_df['y'].mean():.2f}")
                print(f"     결측치: {prophet_df['y'].isna().sum()}개")
                print(f"     최소값: {prophet_df['y'].min():.2f}")
                print(f"     최대값: {prophet_df['y'].max():.2f}")
                
                # 회귀변수별 결측치 확인
                print(f"   회귀변수 결측치 현황:")
                for reg in regressors:
                    if reg in prophet_df.columns:
                        missing = prophet_df[reg].isna().sum()
                        print(f"     {reg}: {missing}개")
                
                raise e  # ✅ 에러 발생시키고 중단

        print(f"   {success_count}/{len(branches)}개 지사 훈련 완료")
        
        if success_count == 0:
            raise RuntimeError("모든 지사에서 Prophet 훈련 실패!")

    def predict(self, df):
        """예측 실행"""
        if len(self.models) == 0:
            raise RuntimeError("훈련된 Prophet 모델이 없습니다!")
        
        predictions = []
        
        # 시즌별 피쳐 설정 사용
        feature_config = model_features[f'prophet_{self.season_type}']
        regressors = feature_config['basic'] + feature_config['seasonal']
        
        for branch in df['branch_id'].unique():
            if branch not in self.models:
                print(f"⚠️ 지사 {branch}: 훈련된 모델 없음, 0으로 채움")
                predictions.extend([0] * len(df[df['branch_id'] == branch]))
                continue

            branch_data = df[df['branch_id'] == branch].copy()
            
            try:
                future_df = pd.DataFrame({'ds': branch_data['tm']})

                # 회귀변수 추가
                for reg in regressors:
                    if reg in branch_data.columns:
                        future_df[reg] = branch_data[reg].values

                forecast = self.models[branch].predict(future_df)
                branch_predictions = np.maximum(forecast['yhat'].values, 0)
                predictions.extend(branch_predictions)
                
            except Exception as e:
                print(f"❌ 지사 {branch} Prophet 예측 실패:")
                print(f"   에러: {str(e)}")
                raise e

        return np.array(predictions)

print("Prophet 최적화 모델 클래스 정의 완료")

Prophet 최적화 모델 클래스 정의 완료


In [17]:
class CatBoostOptimizedModel:
    def __init__(self, season_type="heating"):
        self.model = None
        self.feature_cols = None
        self.categorical_features = None
        self.best_params = {}
        self.season_type = season_type
        
        # 시즌별 피쳐 설정
        feature_config = model_features[f'catboost_{season_type}']
        self.features = feature_config
        
    def prepare_features(self, df):
        """피쳐 준비 및 전처리"""
        df = df.copy()
        
        # 모든 피쳐 수집 (seasonal 추가)
        all_features = []
        for ftype in ['numerical', 'categorical', 'flags', 'seasonal']:
            if ftype in self.features:
                all_features.extend(self.features[ftype])
        
        # 사용 가능한 피쳐만 선택
        available_features = [col for col in all_features if col in df.columns]
        missing_features = [col for col in all_features if col not in df.columns]
        
        if missing_features:
            print(f"⚠️ 누락된 피쳐 ({len(missing_features)}개): {missing_features}")
        
        self.feature_cols = available_features
        
        # 범주형 피쳐 처리
        categorical_features = []
        if 'categorical' in self.features:
            categorical_features = [col for col in self.features['categorical'] if col in df.columns]
            
        self.categorical_features = categorical_features
        
        # 범주형 변수를 문자열로 변환
        for col in categorical_features:
            if col in df.columns:
                df[col] = df[col].astype(str)
        
        print(f"   최종 사용 피쳐: {len(self.feature_cols)}개")
        print(f"   범주형 피쳐: {len(categorical_features)}개 - {categorical_features}")
        
        return df[self.feature_cols], categorical_features
    
    # 단조성 제약 설정 함수 추가
    def _get_monotone_constraints(self, feature_names):
        """난방 수요 예측에 맞는 단조성 제약 설정"""
        constraints = []
        
        for feature in feature_names:
            # ✅ 2. 단조성 제약 설정 (물리적 상식 반영)
            if ('ta' in feature or 'apparent_temp' in feature) and 'lag' not in feature and 'diff' not in feature:  
                # 온도, 체감온도: 온도 ↓ = 난방 수요 ↑ (음의 상관)
                constraints.append(-1)
            elif feature in ['HDD18']:  
                # 난방도일: HDD ↑ = 난방 수요 ↑ (양의 상관)
                constraints.append(1)
            else:  
                # 나머지는 제약 없음 (범주형 변수 cold_extreme 포함)
                constraints.append(0)
        
        constrained_count = sum(1 for c in constraints if c != 0)
        print(f"   단조성 제약: {constrained_count}개 피쳐에 적용")
        return constraints

    def optimize_hyperparameters(self, df, cv_splits, target_col='heat_demand', n_trials=50):
        """연도 기반 CV를 사용한 하이퍼파라미터 최적화"""
        print(f"CatBoost Huber Loss 하이퍼파라미터 최적화 중... (trials: {n_trials})")
        
        # 피쳐 준비
        X_full, categorical_features = self.prepare_features(df)
        y_full = df[target_col].values
        
        print(f"   최적화 데이터: {X_full.shape}")
        print(f"   타겟 통계: 평균={y_full.mean():.2f}, 표준편차={y_full.std():.2f}")
        
        def objective(trial):
            # 하이퍼파라미터 샘플링
            params = {
                'iterations': trial.suggest_int('iterations', 500, 2000),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 20),
                'border_count': trial.suggest_int('border_count', 32, 255),
                'random_seed': SEED,
                'task_type': 'CPU',
                'verbose': 0,
                'loss_function': 'Huber:delta=1.0'  # CatBoost 내장 Huber Loss
            }
            
            cv_scores = []
            
            # 연도 기반 3-Fold CV
            for fold, (train_idx, val_idx) in enumerate(cv_splits):
                try:
                    X_train, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
                    y_train, y_val = y_full[train_idx], y_full[val_idx]
                    
                    # 데이터 크기 확인
                    if len(X_train) < 10 or len(X_val) < 5:
                        print(f"     Fold {fold+1}: 데이터 부족 (train={len(X_train)}, val={len(X_val)})")
                        continue
                    
                    # CatBoost 모델 생성 및 훈련
                    model = CatBoostRegressor(**params, cat_features=categorical_features)
                    model.fit(X_train, y_train, verbose=0)
                    
                    # 예측 및 평가
                    predictions = model.predict(X_val)
                    predictions = np.maximum(predictions, 0)  # 음수 제거
                    
                    # 평가 지표 계산
                    metrics = evaluate_predictions(y_val, predictions, delta=1.0)
                    print(f"     Fold {fold+1}: RMSE={metrics['rmse']:.4f}, Huber={metrics['huber']:.4f}")
                    
                    cv_scores.append(metrics['huber'])  # 최적화는 Huber Loss 기준
                    
                except Exception as e:
                    print(f"❌ Fold {fold+1} CatBoost 최적화 실패:")
                    print(f"   에러: {str(e)}")
                    print(f"   훈련 데이터: {len(X_train) if 'X_train' in locals() else 'N/A'}")
                    print(f"   검증 데이터: {len(X_val) if 'X_val' in locals() else 'N/A'}")
                    raise e  # ✅ 에러 발생시키고 중단
            
            if len(cv_scores) == 0:
                print(f"   ⚠️ 모든 Fold에서 CatBoost 최적화 실패")
                return 999.0
                
            return np.mean(cv_scores)
        
        # Optuna 최적화 실행
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_params = study.best_params
        print(f"   CatBoost 최적 Huber Loss: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_params}")
        return study.best_value

    def fit(self, df, target_col='heat_demand'):
        """최적화된 파라미터로 전체 데이터 훈련"""
        print(f"CatBoost 모델 훈련 중...")
        
        # 피쳐 준비
        X, categorical_features = self.prepare_features(df)
        y = df[target_col].values
        
        # 데이터 품질 확인
        print(f"   훈련 데이터: {X.shape}")
        print(f"   타겟 통계: 평균={y.mean():.2f}, 표준편차={y.std():.2f}, 범위=[{y.min():.2f}, {y.max():.2f}]")
        
        # 결측치 확인
        missing_info = {}
        for col in X.columns:
            missing_count = X[col].isna().sum()
            if missing_count > 0:
                missing_info[col] = missing_count
        
        if missing_info:
            print(f"   ⚠️ 피쳐별 결측치: {missing_info}")
            # 결측치가 있는 경우 처리
            for col, missing_count in missing_info.items():
                if col in categorical_features:
                    X[col] = X[col].fillna('missing')
                else:
                    X[col] = X[col].fillna(X[col].median())
            print(f"   결측치 처리 완료")

        try:
            # ✅ 단조성 제약 계산 ###############################################################
            monotone_constraints = self._get_monotone_constraints(X.columns)
            # 최적화된 파라미터로 모델 생성
            self.model = CatBoostRegressor(
                iterations=self.best_params.get('iterations', 1000),
                learning_rate=self.best_params.get('learning_rate', 0.1),
                depth=self.best_params.get('depth', 6),
                l2_leaf_reg=self.best_params.get('l2_leaf_reg', 3),
                border_count=self.best_params.get('border_count', 128),
                cat_features=categorical_features,
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False,
                loss_function='Huber:delta=1.0',  # Huber Loss 사용
                # ✅ 단조성 제약 추가 적용 ###############################################################
                monotone_constraints=monotone_constraints
            )

            # 모델 훈련
            self.model.fit(X, y)
            print(f"   CatBoost 훈련 완료")
            
            # 피쳐 중요도 출력 (상위 10개)
            if hasattr(self.model, 'feature_importances_'):
                feature_importance = dict(zip(X.columns, self.model.feature_importances_))
                top_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:10]
                print(f"   상위 10개 중요 피쳐:")
                for i, (feature, importance) in enumerate(top_features, 1):
                    print(f"     {i:2d}. {feature}: {importance:.3f}")
        
        except Exception as e:
            print(f"❌ CatBoost 훈련 실패:")
            print(f"   에러: {str(e)}")
            print(f"   데이터 크기: {X.shape}")
            print(f"   범주형 피쳐: {categorical_features}")
            print(f"   파라미터: {self.best_params}")
            raise e  # ✅ 에러 발생시키고 중단

    def predict(self, df):
        """예측 실행"""
        if self.model is None:
            raise RuntimeError("훈련된 CatBoost 모델이 없습니다!")
            
        try:
            # 피쳐 준비
            X, categorical_features = self.prepare_features(df)
            
            # 결측치 처리 (훈련 시와 동일하게)
            for col in X.columns:
                if X[col].isna().sum() > 0:
                    if col in categorical_features:
                        X[col] = X[col].fillna('missing')
                    else:
                        X[col] = X[col].fillna(X[col].median())
            
            predictions = self.model.predict(X)
            predictions = np.maximum(predictions, 0)  # 음수 제거
            
            print(f"   CatBoost 예측 완료: {len(predictions)}개")
            print(f"   예측 통계: 평균={predictions.mean():.2f}, 범위=[{predictions.min():.2f}, {predictions.max():.2f}]")
            
            return predictions
            
        except Exception as e:
            print(f"❌ CatBoost 예측 실패:")
            print(f"   에러: {str(e)}")
            print(f"   데이터 크기: {df.shape}")
            raise e  # ✅ 에러 발생시키고 중단

print("CatBoost 최적화 모델 클래스 정의 완료")

CatBoost 최적화 모델 클래스 정의 완료


In [18]:
class LSTMOptimizedModel:
    def __init__(self, season_type="heating"):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoder = None
        self.device = device
        self.feature_cols = None
        self.best_params = {}
        self.season_type = season_type
        
    def prepare_features(self, df, is_train=True):
        """LSTM용 피쳐 준비"""
        df = df.copy()
        
        # 시즌별 피쳐 설정 사용
        feature_config = model_features[f'lstm_{self.season_type}']
        self.feature_cols = [col for col in feature_config['numerical'] if col in df.columns]
        
        # 누락된 피쳐 확인
        missing_features = [col for col in feature_config['numerical'] if col not in df.columns]
        if missing_features:
            print(f"⚠️ LSTM 누락된 피쳐: {missing_features}")
        
        # 브랜치 인코딩
        if is_train:
            self.label_encoder = LabelEncoder()
            df['branch_encoded'] = self.label_encoder.fit_transform(df['branch_id'])
        else:
            if self.label_encoder is not None:
                try:
                    df['branch_encoded'] = self.label_encoder.transform(df['branch_id'])
                except ValueError as e:
                    print(f"⚠️ 새로운 branch_id 발견, 0으로 처리: {e}")
                    df['branch_encoded'] = 0
            else:
                df['branch_encoded'] = 0
        
        self.feature_cols.append('branch_encoded')
        
        # 결측치 처리
        X = df[self.feature_cols].values
        X = np.nan_to_num(X, nan=0)
        
        print(f"   LSTM 피쳐 준비 완료: {len(self.feature_cols)}개 피쳐")
        
        return X, df

    def optimize_hyperparameters(self, df, cv_splits, target_col='heat_demand', n_trials=30):
        """연도 기반 CV를 사용한 하이퍼파라미터 최적화"""
        print(f"LSTM Huber Loss 하이퍼파라미터 최적화 중... (trials: {n_trials})")
        
        # 피쳐 준비
        X_full, df_processed = self.prepare_features(df, is_train=True)
        y_full = df_processed[target_col].values
        y_full = np.nan_to_num(y_full, nan=0)
        
        print(f"   LSTM 최적화 데이터: {X_full.shape}")
        print(f"   사용 피쳐: {self.feature_cols}")
        
        def objective(trial):
            # 하이퍼파라미터 샘플링
            hidden_size = trial.suggest_int('hidden_size', 64, 256)
            num_layers = trial.suggest_int('num_layers', 1, 3)
            dropout = trial.suggest_float('dropout', 0.1, 0.5)
            learning_rate = trial.suggest_float('learning_rate', 0.0001, 0.01, log=True)
            batch_size = trial.suggest_categorical('batch_size', [32, 64, 128])
            sequence_length = trial.suggest_int('sequence_length', 12, 48)
            
            cv_scores = []
            
            # 연도 기반 3-Fold CV
            for fold, (train_idx, val_idx) in enumerate(cv_splits):
                try:
                    X_train, X_val = X_full[train_idx], X_full[val_idx]
                    y_train, y_val = y_full[train_idx], y_full[val_idx]
                    
                    # 데이터 크기 확인
                    if len(X_train) < sequence_length * 2 or len(X_val) < sequence_length:
                        print(f"     Fold {fold+1}: 데이터 부족 (train={len(X_train)}, val={len(X_val)})")
                        continue
                    
                    # 스케일링
                    scaler = StandardScaler()
                    X_train_scaled = scaler.fit_transform(X_train)
                    X_val_scaled = scaler.transform(X_val)
                    
                    # 데이터셋 생성
                    train_dataset = TimeSeriesDataset(X_train_scaled, y_train, sequence_length)
                    val_dataset = TimeSeriesDataset(X_val_scaled, y_val, sequence_length)
                    
                    if len(train_dataset) == 0 or len(val_dataset) == 0:
                        print(f"     Fold {fold+1}: 시퀀스 데이터셋 생성 실패")
                        continue
                        
                    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
                    
                    # 모델 생성
                    model = LSTMNet(
                        input_size=X_train_scaled.shape[1],
                        hidden_size=hidden_size,
                        num_layers=num_layers,
                        dropout=dropout
                    ).to(self.device)
                    
                    criterion = HuberLoss(delta=1.0)  # ✅ Huber Loss 사용
                    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
                    
                    # 간단한 훈련 (최적화를 위해 짧게)
                    model.train()
                    for epoch in range(10):
                        epoch_loss = 0
                        for batch_x, batch_y in train_loader:
                            batch_x = batch_x.to(self.device)
                            batch_y = batch_y.to(self.device)
                            
                            optimizer.zero_grad()
                            outputs = model(batch_x)
                            loss = criterion(outputs, batch_y)
                            loss.backward()
                            optimizer.step()
                            
                            epoch_loss += loss.item()
                    
                    # 검증
                    model.eval()
                    val_predictions = []
                    val_targets = []
                    
                    with torch.no_grad():
                        for batch_x, batch_y in val_loader:
                            batch_x = batch_x.to(self.device)
                            batch_y = batch_y.to(self.device)
                            
                            outputs = model(batch_x)
                            val_predictions.extend(outputs.cpu().numpy().flatten())
                            val_targets.extend(batch_y.cpu().numpy().flatten())
                    
                    if len(val_predictions) > 0:
                        # Huber Loss 계산
                        huber = huber_score(val_targets, val_predictions, delta=1.0)
                        cv_scores.append(huber)
                        print(f"     Fold {fold+1}: Huber Loss = {huber:.4f} ({len(val_predictions)}개 예측)")
                    
                    # GPU 메모리 정리
                    del model, criterion, optimizer
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        
                except Exception as e:
                    print(f"❌ Fold {fold+1} LSTM 최적화 실패:")
                    print(f"   에러: {str(e)}")
                    raise e  # ✅ 에러 발생시키고 중단
                    
            if len(cv_scores) == 0:
                print(f"   ⚠️ 모든 Fold에서 LSTM 최적화 실패")
                return 999.0
                
            return np.mean(cv_scores)
        
        # Optuna 최적화 실행
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_params = study.best_params
        print(f"   LSTM 최적 Huber Loss: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_params}")
        
        return study.best_value

    def fit(self, df, target_col='heat_demand'):
        """최적화된 파라미터로 전체 데이터 훈련"""
        print(f"LSTM 모델 훈련 중...")
        
        # 피쳐 준비
        X, df_processed = self.prepare_features(df, is_train=True)
        y = df_processed[target_col].values
        y = np.nan_to_num(y, nan=0)

        print(f"   LSTM 훈련 데이터: {X.shape}")
        print(f"   타겟 통계: 평균={y.mean():.2f}, 표준편차={y.std():.2f}")

        # 스케일링
        X_scaled = self.scaler.fit_transform(X)

        # 최적화된 파라미터 사용
        sequence_length = self.best_params.get('sequence_length', 24)
        dataset = TimeSeriesDataset(X_scaled, y, sequence_length)
        
        if len(dataset) == 0:
            raise RuntimeError(f"LSTM 데이터셋이 비어있습니다. 시퀀스 길이: {sequence_length}, 데이터 크기: {len(X_scaled)}")
            
        dataloader = DataLoader(
            dataset, 
            batch_size=self.best_params.get('batch_size', 64), 
            shuffle=True
        )

        try:
            # 모델 생성
            self.model = LSTMNet(
                input_size=X_scaled.shape[1],
                hidden_size=self.best_params.get('hidden_size', 128),
                num_layers=self.best_params.get('num_layers', 2),
                dropout=self.best_params.get('dropout', 0.2)
            ).to(self.device)
            
            criterion = HuberLoss(delta=1.0)  # ✅ Huber Loss 사용
            optimizer = optim.Adam(
                self.model.parameters(), 
                lr=self.best_params.get('learning_rate', 0.001)
            )

            # 훈련
            self.model.train()
            print(f"   시퀀스 길이: {sequence_length}, 배치 크기: {self.best_params.get('batch_size', 64)}")
            
            for epoch in tqdm(range(100), desc="LSTM 훈련"):
                epoch_loss = 0
                batch_count = 0
                
                for batch_x, batch_y in dataloader:
                    batch_x = batch_x.to(self.device)
                    batch_y = batch_y.to(self.device)

                    optimizer.zero_grad()
                    outputs = self.model(batch_x)
                    loss = criterion(outputs, batch_y)
                    loss.backward()
                    optimizer.step()
                    
                    epoch_loss += loss.item()
                    batch_count += 1
                
                # 주기적으로 손실 출력
                if (epoch + 1) % 20 == 0:
                    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0
                    print(f"   Epoch {epoch+1}: 평균 손실 = {avg_loss:.4f}")
                
        except Exception as e:
            print(f"❌ LSTM 훈련 실패:")
            print(f"   에러: {str(e)}")
            print(f"   데이터 크기: {X_scaled.shape}")
            print(f"   시퀀스 길이: {sequence_length}")
            print(f"   파라미터: {self.best_params}")
            raise e  # ✅ 에러 발생시키고 중단
                
        print(f"   LSTM 훈련 완료")

    def predict(self, df):
        """예측 실행"""
        if self.model is None:
            raise RuntimeError("훈련된 LSTM 모델이 없습니다!")

        try:
            # 피쳐 준비
            X, _ = self.prepare_features(df, is_train=False)
            X_scaled = self.scaler.transform(X)

            self.model.eval()
            predictions = []
            sequence_length = self.best_params.get('sequence_length', 24)

            print(f"   LSTM 예측 중... (시퀀스 길이: {sequence_length})")

            with torch.no_grad():
                for i in range(len(X_scaled)):
                    if i < sequence_length:
                        predictions.append(0)  # 초기 시퀀스는 0으로 채움
                    else:
                        seq_data = X_scaled[i-sequence_length+1:i+1]
                        seq_tensor = torch.FloatTensor(seq_data).unsqueeze(0).to(self.device)
                        pred = self.model(seq_tensor).cpu().numpy()[0, 0]
                        predictions.append(max(0, pred))  # 음수 제거

            predictions = np.array(predictions)
            print(f"   LSTM 예측 완료: {len(predictions)}개")
            print(f"   예측 통계: 평균={predictions.mean():.2f}, 범위=[{predictions.min():.2f}, {predictions.max():.2f}]")
            
            return predictions
            
        except Exception as e:
            print(f"❌ LSTM 예측 실패:")
            print(f"   에러: {str(e)}")
            print(f"   데이터 크기: {df.shape}")
            raise e  # ✅ 에러 발생시키고 중단

print("LSTM 최적화 모델 클래스 정의 완료")

LSTM 최적화 모델 클래스 정의 완료


## 7. 스태킹 앙상블 클래스 (Ridge 메타모델 최적화)

In [19]:
from sklearn.linear_model import Ridge

class AdvancedStackingEnsemble:
    def __init__(self, season_type="heating", group_name=""):
        self.season_type = season_type
        self.group_name = group_name
        self.models = {
            'prophet': ProphetOptimizedModel(season_type),
            'catboost': CatBoostOptimizedModel(season_type),
            'lstm': LSTMOptimizedModel(season_type)
        }
        self.meta_model = None
        self.best_meta_params = {}
        self.individual_scores = {}

    def optimize_meta_model(self, level1_features, targets, cv_splits, n_trials=20):
        """Ridge 메타모델 최적화 (연도 기반 CV)"""
        print(f"Ridge 메타모델 최적화 중... (trials: {n_trials})")
        
        def objective(trial):
            # Ridge 파라미터 샘플링
            alpha = trial.suggest_float('alpha', 0.01, 100.0, log=True)
            
            cv_scores = []
            
            # 연도 기반 3-Fold CV
            for fold, (train_idx, val_idx) in enumerate(cv_splits):
                try:
                    # 레벨1 피쳐에서 해당 인덱스 선택
                    train_meta_mask = np.isin(range(len(level1_features)), train_idx)
                    val_meta_mask = np.isin(range(len(level1_features)), val_idx)
                    
                    X_train_meta = level1_features[train_meta_mask]
                    X_val_meta = level1_features[val_meta_mask]
                    y_train_meta = targets[train_meta_mask]
                    y_val_meta = targets[val_meta_mask]
                    
                    if len(X_train_meta) < 5 or len(X_val_meta) < 2:
                        print(f"     메타 Fold {fold+1}: 데이터 부족")
                        continue
                    
                    # Ridge 훈련
                    model = Ridge(alpha=alpha, random_state=SEED)
                    model.fit(X_train_meta, y_train_meta)
                    
                    # 예측 및 평가
                    pred = model.predict(X_val_meta)
                    pred = np.maximum(pred, 0)  # 음수 제거
                    
                    rmse = np.sqrt(mean_squared_error(y_val_meta, pred))
                    cv_scores.append(rmse)
                    print(f"     메타 Fold {fold+1}: RMSE = {rmse:.4f}")
                    
                except Exception as e:
                    print(f"❌ 메타 Fold {fold+1} 최적화 실패:")
                    print(f"   에러: {str(e)}")
                    raise e  # ✅ 에러 발생시키고 중단
                    
            if len(cv_scores) == 0:
                print(f"   ⚠️ 모든 메타 Fold에서 최적화 실패")
                return 999.0
                
            return np.mean(cv_scores)
        
        # Optuna 최적화 실행
        study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
        
        self.best_meta_params = study.best_params
        print(f"   Ridge 최적 RMSE: {study.best_value:.4f}")
        print(f"   최적 파라미터: {self.best_meta_params}")
        
        return study.best_value

    def fit(self, train_df, cv_splits, target_col='heat_demand', optimize_trials=None):
        """스태킹 앙상블 훈련 (연도 기반 CV 사용)"""
        print(f"\n{self.group_name} 스태킹 앙상블 훈련 시작!")
        print("=" * 60)
        
        if len(train_df) < 100:
            raise RuntimeError(f"데이터가 부족합니다 ({len(train_df)}개). 최소 100개 필요.")
            
        # 기본 trials 설정
        if optimize_trials is None:
            optimize_trials = {'prophet': 30, 'catboost': 50, 'lstm': 30, 'meta': 20}

        print(f"훈련 데이터: {len(train_df):,}개")
        print(f"연도 분포: {dict(train_df['year'].value_counts().sort_index())}")

        # 1단계: 개별 모델 하이퍼파라미터 최적화 및 훈련
        level1_predictions_dict = {}

        for name, model in self.models.items():
            print(f"\n{name.upper()} 최적화 및 훈련...")
            try:
                start_time = datetime.now()
                
                # 하이퍼파라미터 최적화 (CV 기반)
                best_score = model.optimize_hyperparameters(
                    train_df, cv_splits, target_col, n_trials=optimize_trials[name]
                )
                
                # 최적화된 파라미터로 전체 훈련 데이터에 재훈련
                model.fit(train_df, target_col)
                
                # CV를 통한 레벨1 예측값 생성 (Out-of-Fold 예측)
                oof_predictions = np.zeros(len(train_df))
                
                for fold, (train_idx, val_idx) in enumerate(cv_splits):
                    print(f"   OOF Fold {fold+1} 처리 중...")
                    
                    fold_train = train_df.iloc[train_idx]
                    fold_val = train_df.iloc[val_idx]
                    
                    # 폴드별 모델 훈련 (최적 파라미터 사용)
                    if name == 'prophet':
                        fold_model = ProphetOptimizedModel(self.season_type)
                        fold_model.best_params = model.best_params
                        fold_model.fit(fold_train, target_col)
                        fold_pred = fold_model.predict(fold_val)
                    elif name == 'catboost':
                        fold_model = CatBoostOptimizedModel(self.season_type)
                        fold_model.best_params = model.best_params
                        fold_model.fit(fold_train, target_col)
                        fold_pred = fold_model.predict(fold_val)
                    else:  # lstm
                        fold_model = LSTMOptimizedModel(self.season_type)
                        fold_model.best_params = model.best_params
                        # LSTM은 CV splits를 직접 전달하지 않고 fit만 수행
                        fold_model.fit(fold_train, target_col)
                        fold_pred = fold_model.predict(fold_val)
                    
                    oof_predictions[val_idx] = fold_pred
                    
                    # GPU 메모리 정리 (LSTM의 경우)
                    if name == 'lstm' and torch.cuda.is_available():
                        torch.cuda.empty_cache()

                level1_predictions_dict[name] = oof_predictions

                # 개별 모델 성능 계산
                metrics = evaluate_predictions(train_df[target_col].values, oof_predictions, delta=1.0)
                mae = mean_absolute_error(train_df[target_col].values, oof_predictions)
                train_time = (datetime.now() - start_time).total_seconds()

                self.individual_scores[name] = {
                    'rmse': metrics['rmse'],
                    'huber': metrics['huber'], 
                    'mae': mae, 
                    'optuna_score': best_score,
                    'train_time': train_time
                }

                print(f"   {name} 성능: RMSE={metrics['rmse']:.4f}, Huber={metrics['huber']:.4f}, MAE={mae:.4f}")
                print(f"   Optuna 최적 점수: {best_score:.4f}")
                print(f"   총 시간: {train_time:.1f}초")

            except Exception as e:
                print(f"❌ {name} 훈련 실패:")
                print(f"   에러: {str(e)}")
                raise e  # ✅ 에러 발생시키고 중단

        # 2단계: 메타 모델 최적화 및 훈련
        print(f"\nRidge 메타 모델 최적화 및 훈련...")
        
        # 레벨1 피쳐 구성
        level1_features = np.column_stack(list(level1_predictions_dict.values()))
        targets = train_df[target_col].values
        
        print(f"   메타 모델 입력: {level1_features.shape}")
        print(f"   개별 모델 예측 통계:")
        for i, (name, pred) in enumerate(level1_predictions_dict.items()):
            print(f"     {name}: 평균={pred.mean():.2f}, 표준편차={pred.std():.2f}")
        
        # 메타모델 하이퍼파라미터 최적화
        meta_score = self.optimize_meta_model(level1_features, targets, cv_splits, optimize_trials['meta'])
        
        # 최적화된 파라미터로 메타모델 훈련
        try:
            self.meta_model = Ridge(
                alpha=self.best_meta_params.get('alpha', 1.0),
                random_state=SEED
            )
            self.meta_model.fit(level1_features, targets)

            # 스태킹 성능 계산
            stacking_pred = self.meta_model.predict(level1_features)
            stacking_pred = np.maximum(stacking_pred, 0)  # 음수 제거
            
            stacking_metrics = evaluate_predictions(targets, stacking_pred, delta=1.0)
            stacking_mae = mean_absolute_error(targets, stacking_pred)

            self.individual_scores['stacking'] = {
                'rmse': stacking_metrics['rmse'],
                'huber': stacking_metrics['huber'], 
                'mae': stacking_mae,
                'optuna_score': meta_score
            }
            
            print(f"   스태킹 성능: RMSE={stacking_metrics['rmse']:.4f}, Huber={stacking_metrics['huber']:.4f}, MAE={stacking_mae:.4f}")
            
            # 메타모델 가중치 출력
            if hasattr(self.meta_model, 'coef_'):
                model_names = list(level1_predictions_dict.keys())
                print(f"   메타모델 가중치:")
                for i, (name, coef) in enumerate(zip(model_names, self.meta_model.coef_)):
                    print(f"     {name}: {coef:.4f}")
            
        except Exception as e:
            print(f"❌ 메타모델 훈련 실패:")
            print(f"   에러: {str(e)}")
            raise e  # ✅ 에러 발생시키고 중단
            
        print(f"✅ {self.group_name} 스태킹 앙상블 훈련 완료!")

    def predict(self, test_df):
        """스태킹 앙상블 예측"""
        if self.meta_model is None:
            raise RuntimeError(f"{self.group_name} 모델이 훈련되지 않았습니다!")

        level1_predictions = {}

        # 1단계: 개별 모델 예측
        print(f"   {self.group_name} 개별 모델 예측 중...")
        for name, model in self.models.items():
            try:
                level1_predictions[name] = model.predict(test_df)
                pred_stats = level1_predictions[name]
                print(f"     {name}: 평균={pred_stats.mean():.2f}, 범위=[{pred_stats.min():.2f}, {pred_stats.max():.2f}]")
            except Exception as e:
                print(f"❌ {name} 예측 실패:")
                print(f"   에러: {str(e)}")
                raise e  # ✅ 에러 발생시키고 중단

        # 2단계: 메타 모델 예측
        try:
            meta_features = np.column_stack(list(level1_predictions.values()))
            final_pred = self.meta_model.predict(meta_features)
            final_pred = np.maximum(final_pred, 0)  # 음수 제거

            print(f"   {self.group_name} 스태킹 예측 완료: 평균={final_pred.mean():.2f}, 범위=[{final_pred.min():.2f}, {final_pred.max():.2f}]")
            
            return final_pred, level1_predictions
            
        except Exception as e:
            print(f"❌ {self.group_name} 메타모델 예측 실패:")
            print(f"   에러: {str(e)}")
            raise e  # ✅ 에러 발생시키고 중단

print("✅ 고도화된 스태킹 앙상블 클래스 정의 완료")

# 결과 저장용 딕셔너리
ensemble_models = {}
group_results = {}

print("\n🎯 2개 그룹별 개별 훈련 준비 완료!")

✅ 고도화된 스태킹 앙상블 클래스 정의 완료

🎯 2개 그룹별 개별 훈련 준비 완료!


### (디버깅용) 단순 보간 함수, 0으로 일단 채우기

In [ ]:
# for group_name, group_data in train_groups.items():
#     if len(group_data) > 100:
#         cv_splits = create_year_based_cv_splits(group_data, group_name)
#         print(f"   {group_name}: {len(cv_splits)}개 fold 생성됨")
        
#         # apparent_temp와 lag 변수들 결측치를 0으로 채우기
#         lag_cols = ['apparent_temp', 'ta_lag_3h', 'ta_lag_6h', 'ta_lag_24h']
        
#         for col in lag_cols:
#             if col in group_data.columns:
#                 missing_count = group_data[col].isna().sum()
#                 if missing_count > 0:
#                     print(f"     🔧 {col} 결측치 {missing_count}개를 0으로 채움")
#                     train_groups[group_name][col] = group_data[col].fillna(0)
        
#         group_data = train_groups[group_name]  # 업데이트된 데이터로 갱신
        
#         # 결측치 최종 확인
#         weather_cols = ['ta', 'hm', 'ws', 'rn_day', 'rn_hr1', 'si', 'ta_chi', 'apparent_temp']
#         total_missing = 0
#         for col in weather_cols:
#             if col in group_data.columns:
#                 missing = group_data[col].isna().sum()
#                 total_missing += missing
#                 if missing > 0:
#                     print(f"     ⚠️ {col}: {missing}개 결측치 남음")
        
#         if total_missing == 0:
#             print(f"     ✅ {group_name}: 모든 기상변수 결측치 해결됨")
#         print()

non_heating 그룹 - 연도 기반 3-Fold CV 분할 생성...
   연도별 데이터 분포:
     2021년: 69,768개
     2022년: 69,768개
     2023년: 69,768개
   Fold 2021: 훈련 139,536개, 검증 69,768개
   Fold 2022: 훈련 139,536개, 검증 69,768개
   Fold 2023: 훈련 139,536개, 검증 69,768개
   non_heating: 3개 fold 생성됨
     🔧 ta_lag_24h 결측치 24개를 0으로 채움
     ✅ non_heating: 모든 기상변수 결측치 해결됨

heating 그룹 - 연도 기반 3-Fold CV 분할 생성...
   연도별 데이터 분포:
     2021년: 96,653개
     2022년: 96,672개
     2023년: 96,672개
   Fold 2021: 훈련 193,344개, 검증 96,653개
   Fold 2022: 훈련 193,325개, 검증 96,672개
   Fold 2023: 훈련 193,325개, 검증 96,672개
   heating: 3개 fold 생성됨
     🔧 ta_lag_24h 결측치 24개를 0으로 채움
     ✅ heating: 모든 기상변수 결측치 해결됨



## 8. 그룹별 개별 훈련

### (현재) Prophet

"전체 기준으로 하이퍼파라미터를 찾은 후, 지사별로 학습" ✅

하이퍼파라미터: 모든 지사 통합 성능으로 최적화
모델 훈련: 찾은 파라미터로 지사별 개별 모델 생성

2. 실제로는

하나의 파라미터 조합을 모든 지사에 적용해서 테스트
지사별 성능을 종합해서 그 파라미터 조합의 점수 계산
30번 반복해서 가장 좋은 파라미터 조합 찾기

3. 최종 결과

전체적으로 가장 좋은 하나의 파라미터 세트 선택
이 파라미터를 모든 지사에 동일하게 적용해서 개별 모델 훈련

In [32]:
# 모든 그룹 훈련 함수
def train_all_groups():
    """2개 그룹 훈련 (난방/비난방) - 연도 기반 CV 적용"""
    group_configs = {
        "heating": {
            "season": "heating", 
            # "trials": {"prophet": 40, "catboost": 60, "lstm": 40, "meta": 25}
            "trials": {"prophet": 1, "catboost": 1, "lstm": 1, "meta": 1}
        },
        "non_heating": {
            "season": "non_heating", 
            # "trials": {"prophet": 30, "catboost": 50, "lstm": 30, "meta": 20}
            "trials": {"prophet": 1, "catboost": 1, "lstm": 1, "meta": 1}
        }
    }
    
    print("🚀 시즌별 2개 그룹 훈련 시작 (연도 기반 3-Fold CV)")
    total_start_time = datetime.now()
    
    for group_name, config in group_configs.items():
        print(f"\n{'='*60}")
        print(f"🔥 {group_name.upper()} 그룹 훈련")
        
        # 그룹 데이터 검증
        if group_name not in train_groups:
            raise KeyError(f"'{group_name}' 그룹이 train_groups에 없습니다.")
            
        group_data = train_groups[group_name]
        
        if len(group_data) == 0:
            raise ValueError(f"{group_name} 그룹에 데이터가 없습니다.")
        
        if 'heat_demand' not in group_data.columns:
            raise ValueError(f"{group_name} 그룹에 'heat_demand' 컬럼이 없습니다.")
        
        print(f"📊 데이터 크기: {len(group_data):,}개")
        print(f"🏢 지사 수: {group_data['branch_id'].nunique()}개")
        print(f"📅 연도 분포: {dict(group_data['year'].value_counts().sort_index())}")
        print(f"🎯 타겟 통계: 평균={group_data['heat_demand'].mean():.2f}, 표준편차={group_data['heat_demand'].std():.2f}")
        
        # 최소 데이터 요구량 확인
        if len(group_data) < 1000:
            raise ValueError(f"{group_name} 데이터가 부족합니다 ({len(group_data):,}개). 최소 1,000개 필요.")
        
        # 그룹별 CV 분할 생성
        print(f"🔄 {group_name} CV 분할 생성 중...")
        group_cv_splits = create_year_based_cv_splits(group_data, group_name)
        
        # 앙상블 모델 생성
        print(f"🏗️ {group_name} 앙상블 모델 생성 중...")
        ensemble_models[group_name] = AdvancedStackingEnsemble(
            season_type=config["season"], 
            group_name=group_name
        )
        
        # 훈련 실행
        print(f"🚀 {group_name} 훈련 시작...")
        start_time = datetime.now()
        
        ensemble_models[group_name].fit(
            group_data, 
            group_cv_splits,
            target_col='heat_demand',
            optimize_trials=config["trials"]
        )
        
        total_time = (datetime.now() - start_time).total_seconds()
        
        # 결과 저장
        group_results[group_name] = {
            'scores': ensemble_models[group_name].individual_scores.copy(),
            'total_time': total_time,
            'data_size': len(group_data),
            'branch_count': group_data['branch_id'].nunique(),
            'year_distribution': dict(group_data['year'].value_counts().sort_index())
        }
        
        # 성능 결과 출력 (Huber Loss 포함)
        print(f"\n📈 {group_name} 최종 결과:")
        print(f"{'모델':12s} {'RMSE':>8s} {'Huber':>8s} {'MAE':>8s} {'시간(초)':>8s}")
        print("-" * 50)
        
        for model, scores in ensemble_models[group_name].individual_scores.items():
            rmse = scores.get('rmse', 999)
            huber = scores.get('huber', 999)
            mae = scores.get('mae', 999)
            model_time = scores.get('train_time', 0)
            
            print(f"{model:12s} {rmse:8.4f} {huber:8.4f} {mae:8.4f} {model_time:8.1f}")
        
        print(f"   ⏱️ 총 훈련 시간: {total_time:.1f}초 ({total_time/60:.1f}분)")
        
        # 최고 성능 모델 확인
        best_model = min(
            [(name, score.get('huber', 999)) for name, score in ensemble_models[group_name].individual_scores.items()],
            key=lambda x: x[1]
        )
        print(f"   🏆 최고 성능: {best_model[0]} (Huber Loss: {best_model[1]:.4f})")
        print(f"✅ {group_name} 그룹 훈련 완료!")
    
    # 전체 결과 요약
    total_training_time = (datetime.now() - total_start_time).total_seconds()
    
    print(f"\n🎉 전체 훈련 완료!")
    print(f"=" * 60)
    print(f"⏱️ 총 훈련 시간: {total_training_time/60:.1f}분 ({total_training_time/3600:.1f}시간)")
    
    # 전체 평균 성능
    print(f"\n📊 전체 평균 성능:")
    avg_scores = {'rmse': [], 'huber': [], 'mae': []}
    
    for group_name, result in group_results.items():
        if result is not None and 'scores' in result:
            stacking_score = result['scores'].get('stacking', {})
            for metric in avg_scores.keys():
                if metric in stacking_score:
                    avg_scores[metric].append(stacking_score[metric])
    
    for metric, scores in avg_scores.items():
        if scores:
            print(f"   평균 {metric.upper()}: {np.mean(scores):.4f} (±{np.std(scores):.4f})")
    
    # GPU 메모리 정리
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"🧹 GPU 메모리 정리 완료")

# 모든 그룹 훈련 실행
train_all_groups()

🚀 시즌별 2개 그룹 훈련 시작 (연도 기반 3-Fold CV)

🔥 HEATING 그룹 훈련
📊 데이터 크기: 289,997개
🏢 지사 수: 19개
📅 연도 분포: {2021: 96653, 2022: 96672, 2023: 96672}
🎯 타겟 통계: 평균=135.94, 표준편차=135.30
🔄 heating CV 분할 생성 중...
heating 그룹 - 연도 기반 3-Fold CV 분할 생성...
   연도별 데이터 분포:
     2021년: 96,653개
     2022년: 96,672개
     2023년: 96,672개
   Fold 2021: 훈련 193,344개, 검증 96,653개
   Fold 2022: 훈련 193,325개, 검증 96,672개
   Fold 2023: 훈련 193,325개, 검증 96,672개
🏗️ heating 앙상블 모델 생성 중...
🚀 heating 훈련 시작...

heating 스태킹 앙상블 훈련 시작!
훈련 데이터: 289,997개
연도 분포: {2021: 96653, 2022: 96672, 2023: 96672}

PROPHET 최적화 및 훈련...
Prophet Huber Loss 하이퍼파라미터 최적화 중... (trials: 1)


  0%|          | 0/1 [00:00<?, ?it/s]

00:33:23 - cmdstanpy - INFO - Chain [1] start processing
00:33:25 - cmdstanpy - INFO - Chain [1] done processing
00:33:26 - cmdstanpy - INFO - Chain [1] start processing
00:33:28 - cmdstanpy - INFO - Chain [1] done processing
00:33:29 - cmdstanpy - INFO - Chain [1] start processing
00:33:31 - cmdstanpy - INFO - Chain [1] done processing
00:33:32 - cmdstanpy - INFO - Chain [1] start processing
00:33:33 - cmdstanpy - INFO - Chain [1] done processing
00:33:34 - cmdstanpy - INFO - Chain [1] start processing
00:33:36 - cmdstanpy - INFO - Chain [1] done processing
00:33:37 - cmdstanpy - INFO - Chain [1] start processing
00:33:39 - cmdstanpy - INFO - Chain [1] done processing
00:33:40 - cmdstanpy - INFO - Chain [1] start processing
00:33:42 - cmdstanpy - INFO - Chain [1] done processing
00:33:43 - cmdstanpy - INFO - Chain [1] start processing
00:33:46 - cmdstanpy - INFO - Chain [1] done processing
00:33:47 - cmdstanpy - INFO - Chain [1] start processing
00:33:48 - cmdstanpy - INFO - Chain [1]

   Fold 1: Huber Loss = 23.5652 (96653개 예측)


00:34:18 - cmdstanpy - INFO - Chain [1] start processing
00:34:20 - cmdstanpy - INFO - Chain [1] done processing
00:34:21 - cmdstanpy - INFO - Chain [1] start processing
00:34:23 - cmdstanpy - INFO - Chain [1] done processing
00:34:24 - cmdstanpy - INFO - Chain [1] start processing
00:34:26 - cmdstanpy - INFO - Chain [1] done processing
00:34:27 - cmdstanpy - INFO - Chain [1] start processing
00:34:28 - cmdstanpy - INFO - Chain [1] done processing
00:34:30 - cmdstanpy - INFO - Chain [1] start processing
00:34:31 - cmdstanpy - INFO - Chain [1] done processing
00:34:32 - cmdstanpy - INFO - Chain [1] start processing
00:34:34 - cmdstanpy - INFO - Chain [1] done processing
00:34:35 - cmdstanpy - INFO - Chain [1] start processing
00:34:36 - cmdstanpy - INFO - Chain [1] done processing
00:34:38 - cmdstanpy - INFO - Chain [1] start processing
00:34:39 - cmdstanpy - INFO - Chain [1] done processing
00:34:40 - cmdstanpy - INFO - Chain [1] start processing
00:34:42 - cmdstanpy - INFO - Chain [1]

...

00:40:15 - cmdstanpy - INFO - Chain [1] start processing
00:40:17 - cmdstanpy - INFO - Chain [1] done processing


   19/19개 지사 훈련 완료
   prophet 성능: RMSE=36.6158, Huber=23.3987, MAE=23.8899
   Optuna 최적 점수: 23.3987
   총 시간: 428.6초

CATBOOST 최적화 및 훈련...
CatBoost Huber Loss 하이퍼파라미터 최적화 중... (trials: 1)
⚠️ 누락된 피쳐 (3개): ['HDD18hour_sin', 'heating_month_sin', 'heating_month_cos']
   최종 사용 피쳐: 47개
   범주형 피쳐: 9개 - ['branch_id', 'hour_cat', 'month_cat', 'weekday_name', 'temp_category', 'wind_category', 'holiday_type', 'peak_time', 'cold_warning_level']
   최적화 데이터: (289997, 47)
   타겟 통계: 평균=135.94, 표준편차=135.30


  0%|          | 0/1 [00:00<?, ?it/s]

     Fold 1: RMSE=27.3039, Huber=15.5557
     Fold 2: RMSE=25.2624, Huber=14.8776
     Fold 3: RMSE=25.2278, Huber=15.0983
   CatBoost 최적 Huber Loss: 15.1772
   최적 파라미터: {'iterations': 1062, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 12.374511199743695, 'border_count': 66}
CatBoost 모델 훈련 중...
⚠️ 누락된 피쳐 (3개): ['HDD18hour_sin', 'heating_month_sin', 'heating_month_cos']
   최종 사용 피쳐: 47개
   범주형 피쳐: 9개 - ['branch_id', 'hour_cat', 'month_cat', 'weekday_name', 'temp_category', 'wind_category', 'holiday_type', 'peak_time', 'cold_warning_level']
   훈련 데이터: (289997, 47)
   타겟 통계: 평균=135.94, 표준편차=135.30, 범위=[-99.00, 966.00]
   단조성 제약: 11개 피쳐에 적용
   CatBoost 훈련 완료
   상위 10개 중요 피쳐:
      1. branch_id: 31.142
      2. dayofweek_sin: 9.880
      3. hour_cos: 7.569
      4. heating_month_order: 7.462
      5. daily_ta_mean: 5.236
      6. ta_lag_3h: 4.310
      7. dayofyear: 3.885
      8. dayofweek_cos: 3.475
      9. daily_temp_range: 3.382
     10. ta_diff_6h: 3.369
   OOF Fol

KeyError: 'lstm'

## 9. 전체 그룹 결과 요약

In [ ]:
# 전체 그룹 훈련 결과 요약
print("\n🏆 전체 그룹 훈련 결과 요약")
print("=" * 120)

total_time = 0
total_data_size = 0
successful_groups = 0

# 헤더 출력 (RMSE/Huber 형태로)
print(f"{'그룹명':15s} {'데이터':>8s} {'Prophet':>15s} {'CatBoost':>15s} {'LSTM':>15s} {'Stacking':>15s} {'시간(분)':>8s}")
print(f"{'':15s} {'':>8s} {'RMSE/Huber':>15s} {'RMSE/Huber':>15s} {'RMSE/Huber':>15s} {'RMSE/Huber':>15s} {'':>8s}")
print("-" * 120)

for group_name, result in group_results.items():
    if result is not None:
        scores = result['scores']
        data_size = result['data_size']
        group_time = result['total_time']
        
        total_time += group_time
        total_data_size += data_size
        successful_groups += 1
        
        # RMSE와 Huber Loss 모두 가져오기
        prophet_rmse = scores.get('prophet', {}).get('rmse', 999)
        prophet_huber = scores.get('prophet', {}).get('huber', 999)
        catboost_rmse = scores.get('catboost', {}).get('rmse', 999)
        catboost_huber = scores.get('catboost', {}).get('huber', 999)
        lstm_rmse = scores.get('lstm', {}).get('rmse', 999)
        lstm_huber = scores.get('lstm', {}).get('huber', 999)
        stacking_rmse = scores.get('stacking', {}).get('rmse', 999)
        stacking_huber = scores.get('stacking', {}).get('huber', 999)
        
        # RMSE/Huber 형태로 출력
        prophet_display = f"{prophet_rmse:.2f}/{prophet_huber:.2f}"
        catboost_display = f"{catboost_rmse:.2f}/{catboost_huber:.2f}"
        lstm_display = f"{lstm_rmse:.2f}/{lstm_huber:.2f}"
        stacking_display = f"{stacking_rmse:.2f}/{stacking_huber:.2f}"
        
        print(f"{group_name:15s} {data_size:8,d} {prophet_display:>15s} {catboost_display:>15s} {lstm_display:>15s} {stacking_display:>15s} {group_time/60:8.1f}")
    else:
        print(f"{group_name:15s} {'N/A':>8s} {'N/A':>15s} {'N/A':>15s} {'N/A':>15s} {'N/A':>15s} {'N/A':>8s}")

print("-" * 120)
print(f"{'TOTAL':15s} {total_data_size:8,d} {'':>15s} {'':>15s} {'':>15s} {'':>15s} {total_time/60:8.1f}")
print(f"\n✅ 성공한 그룹: {successful_groups}/2")
print(f"⏱️ 총 훈련 시간: {total_time/60:.1f}분 ({total_time/3600:.1f}시간)")

# 그룹별 최고 성능 모델 찾기 (Huber Loss 기준)
print(f"\n🥇 그룹별 최고 성능 모델 (Huber Loss 기준):")
for group_name, result in group_results.items():
    if result is not None:
        scores = result['scores']
        best_model = min(
            [(name, score['huber']) for name, score in scores.items() 
             if isinstance(score, dict) and 'huber' in score],
            key=lambda x: x[1],
            default=("None", 999)
        )
        print(f"   {group_name:15s}: {best_model[0].upper():10s} (Huber: {best_model[1]:.4f})")

# 모델별 평균 성능 (RMSE와 Huber 모두)
print(f"\n📊 모델별 평균 성능:")
model_avg_scores = {
    'prophet': {'rmse': [], 'huber': []}, 
    'catboost': {'rmse': [], 'huber': []}, 
    'lstm': {'rmse': [], 'huber': []}, 
    'stacking': {'rmse': [], 'huber': []}
}

for result in group_results.values():
    if result is not None:
        for model_name in model_avg_scores.keys():
            if (model_name in result['scores'] and 
                isinstance(result['scores'][model_name], dict)):
                score_dict = result['scores'][model_name]
                if 'rmse' in score_dict:
                    model_avg_scores[model_name]['rmse'].append(score_dict['rmse'])
                if 'huber' in score_dict:
                    model_avg_scores[model_name]['huber'].append(score_dict['huber'])

print(f"{'모델':12s} {'평균 RMSE':>12s} {'평균 Huber':>12s}")
print("-" * 40)
for model_name, scores in model_avg_scores.items():
    rmse_scores = scores['rmse']
    huber_scores = scores['huber']
    
    if rmse_scores and huber_scores:
        avg_rmse = np.mean(rmse_scores)
        avg_huber = np.mean(huber_scores)
        print(f"{model_name.upper():12s} {avg_rmse:12.4f} {avg_huber:12.4f}")

# 스태킹의 개선 효과 분석
print(f"\n🎯 스태킹 앙상블 개선 효과 (Huber Loss 기준):")
for group_name, result in group_results.items():
    if result is not None:
        scores = result['scores']
        individual_huber_scores = []
        
        for model in ['prophet', 'catboost', 'lstm']:
            if model in scores and 'huber' in scores[model]:
                individual_huber_scores.append(scores[model]['huber'])
        
        if individual_huber_scores and 'stacking' in scores and 'huber' in scores['stacking']:
            best_individual = min(individual_huber_scores)
            stacking_score = scores['stacking']['huber']
            improvement = ((best_individual - stacking_score) / best_individual) * 100
            
            print(f"   {group_name:15s}: {improvement:+6.2f}% 개선")
            print(f"                     (최고 개별: {best_individual:.4f} → 스태킹: {stacking_score:.4f})")

## 🔟 테스트 데이터 예측

In [ ]:
# 테스트 데이터 예측
print("🎯 테스트 데이터 예측 시작...")

# 예측 결과 저장용
test_predictions = {}
individual_predictions = {}

# 최종 예측 결과 통합에서도 2개 그룹만 처리
for group_name in ['heating', 'non_heating']:
    if group_name in ensemble_models and len(test_groups[group_name]) > 0:
        print(f"\n📊 {group_name} 예측 중...")
        
        try:
            pred, individual_pred = ensemble_models[group_name].predict(test_groups[group_name])
            test_predictions[group_name] = pred
            individual_predictions[group_name] = individual_pred
            
            print(f"   ✅ {group_name}: {len(pred):,}개 예측 완료")
            print(f"   📈 예측값 범위: {pred.min():.2f} ~ {pred.max():.2f}")
            print(f"   📊 예측값 평균: {pred.mean():.2f}")
            
        except Exception as e:
            print(f"   ❌ {group_name} 예측 실패: {str(e)[:100]}...")
            test_predictions[group_name] = np.zeros(len(test_groups[group_name]))
    else:
        if len(test_groups[group_name]) > 0:
            print(f"⚠️ {group_name}: 훈련된 모델 없음, 0으로 채움")
            test_predictions[group_name] = np.zeros(len(test_groups[group_name]))

print("\n✅ 모든 그룹 예측 완료!")

## 1️⃣1️⃣ 최종 결과 통합 및 저장

In [ ]:
# 최종 예측 결과 통합
print("💾 최종 예측 결과 통합 및 저장...")

# 기본 결과 데이터프레임 생성
result_df = test_df[['tm', 'branch_id', 'heating_season', 'size_group']].copy()

# 그룹별 예측 결과 통합
final_stacking_pred = np.zeros(len(test_df))
final_prophet_pred = np.zeros(len(test_df))
final_catboost_pred = np.zeros(len(test_df))
final_lstm_pred = np.zeros(len(test_df))

print("📊 그룹별 예측 결과 통합 중...")

# 각 그룹별로 해당하는 인덱스에 예측값 할당
for group_name, group_data in test_groups.items():
    if len(group_data) > 0 and group_name in test_predictions:
        group_indices = group_data.index
        group_pred = test_predictions[group_name]
        
        print(f"   {group_name}: {len(group_indices)}개 인덱스, {len(group_pred)}개 예측값")
        
        # 인덱스 길이 맞추기
        min_length = min(len(group_indices), len(group_pred))
        if min_length > 0:
            final_stacking_pred[group_indices[:min_length]] = group_pred[:min_length]
            
            # 개별 모델 예측값도 저장
            if group_name in individual_predictions:
                individual_pred = individual_predictions[group_name]
                
                if 'prophet' in individual_pred and len(individual_pred['prophet']) >= min_length:
                    final_prophet_pred[group_indices[:min_length]] = individual_pred['prophet'][:min_length]
                if 'catboost' in individual_pred and len(individual_pred['catboost']) >= min_length:
                    final_catboost_pred[group_indices[:min_length]] = individual_pred['catboost'][:min_length]
                if 'lstm' in individual_pred and len(individual_pred['lstm']) >= min_length:
                    final_lstm_pred[group_indices[:min_length]] = individual_pred['lstm'][:min_length]

# 음수값 제거
final_stacking_pred = np.maximum(final_stacking_pred, 0)
final_prophet_pred = np.maximum(final_prophet_pred, 0)
final_catboost_pred = np.maximum(final_catboost_pred, 0)
final_lstm_pred = np.maximum(final_lstm_pred, 0)

# 결과 데이터프레임에 추가
result_df['stacking_prediction'] = final_stacking_pred.round(1)
result_df['prophet_prediction'] = final_prophet_pred.round(1)
result_df['catboost_prediction'] = final_catboost_pred.round(1)
result_df['lstm_prediction'] = final_lstm_pred.round(1)

# 통계 출력
print(f"\n📈 최종 예측값 통계:")
prediction_cols = ['stacking_prediction', 'prophet_prediction', 'catboost_prediction', 'lstm_prediction']
for col in prediction_cols:
    mean_val = result_df[col].mean()
    std_val = result_df[col].std()
    max_val = result_df[col].max()
    min_val = result_df[col].min()
    print(f"   {col:20s}: 평균={mean_val:7.1f}, 표준편차={std_val:6.1f}, 범위=[{min_val:.1f}, {max_val:.1f}]")

# CSV 파일 저장
result_filename = 'advanced_stacking_ensemble_predictions.csv'
result_df.to_csv(result_filename, index=False)
print(f"\n📁 상세 예측 결과 저장: {result_filename}")

# 제출용 파일 생성 (스태킹 앙상블 결과만)
submission_df = test_df[['tm', 'branch_id']].copy()
submission_df['heat_demand'] = result_df['stacking_prediction']

submission_filename = 'submission_advanced_stacking.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"📁 제출용 파일 저장: {submission_filename}")

# 그룹별 예측 통계
print(f"\n📊 그룹별 예측 통계 (스태킹 모델):")
try:
    group_stats = result_df.groupby(['heating_season', 'size_group'])['stacking_prediction'].agg([
        'count', 'mean', 'std', 'min', 'max'
    ]).round(2)
    print(group_stats)
except Exception as e:
    print(f"   그룹별 통계 계산 실패: {e}")

# Google Drive 저장 (Colab 환경)
if IN_COLAB:
    try:
        save_drive = input("\nGoogle Drive에 저장하시겠습니까? (y/n): ").lower().strip()
        if save_drive == 'y':
            os.system(f"cp {result_filename} /content/drive/MyDrive/")
            os.system(f"cp {submission_filename} /content/drive/MyDrive/")
            print("✅ Google Drive 저장 완료!")
    except Exception as e:
        print(f"⚠️ Google Drive 저장 중 오류: {e}")

print("\n🎊 모든 작업 완료!")

## 1️⃣2️⃣ 최종 분석 요약

In [ ]:
print("\n📋 🔥 최종 분석 요약 🔥")
print("=" * 80)

print(f"✅ 모델 구성:")
print(f"   🎯 고도화된 스태킹 앙상블 (Prophet + CatBoost + LSTM + Ridge)")
print(f"   📊 그룹별 전용 모델: 3개 규모 × 2개 시즌 = 6개 그룹")
print(f"   🔬 총 모델 수: 18개 (6그룹 × 3모델) + 6개 메타모델 = 24개")
print(f"   🎲 완전한 재현성: 모든 시드 고정 (SEED={SEED})")

print(f"\n🔧 기술적 혁신:")
print(f"   🌡️ 시즌별 도메인 특화 이상치 플래그")
print(f"   📈 모델별 차별화된 피쳐 전략")
print(f"   🎯 Optuna TPE + 3-Fold CV로 전 모델 최적화")
print(f"   🔄 ARIMA 시계열 보간 (LSTM용)")

print(f"\n📊 모델별 특화 전략:")
print(f"   🔮 Prophet: 시계열 중심, 결측치 내장 처리")
print(f"   🐱 CatBoost: 범주형 최적화, 결측치+이상치 플래그")
print(f"   🧠 LSTM: 수치형 정규화, ARIMA 보간")
print(f"   🎯 Ridge: 최적화된 메타모델")

# 최종 성능 요약
if group_results:
    successful_groups = sum(1 for result in group_results.values() if result is not None)
    total_training_time = sum(
        result['total_time'] for result in group_results.values() 
        if result is not None
    )
    
    print(f"\n🏆 훈련 결과:")
    print(f"   ✅ 성공한 그룹: {successful_groups}/6")
    print(f"   ⏱️ 총 훈련 시간: {total_training_time/60:.1f}분")
    
    # 스태킹 vs 개별 모델 성능 비교
    stacking_scores = []
    individual_scores = {'prophet': [], 'catboost': [], 'lstm': []}
    
    for result in group_results.values():
        if result is not None and 'scores' in result:
            scores = result['scores']
            if ('stacking' in scores and 
                isinstance(scores['stacking'], dict) and 
                'rmse' in scores['stacking']):
                stacking_scores.append(scores['stacking']['rmse'])
            
            for model in individual_scores.keys():
                if (model in scores and 
                    isinstance(scores[model], dict) and 
                    'rmse' in scores[model]):
                    individual_scores[model].append(scores[model]['rmse'])
    
    if stacking_scores:
        print(f"\n📈 평균 성능 (RMSE):")
        for model, scores in individual_scores.items():
            if scores:
                avg_score = np.mean(scores)
                print(f"   {model.upper():12s}: {avg_score:.4f}")
        
        stacking_avg = np.mean(stacking_scores)
        print(f"   {'STACKING':12s}: {stacking_avg:.4f} ⭐")
        
        # 개선율 계산
        individual_avgs = [np.mean(scores) for scores in individual_scores.values() if scores]
        if individual_avgs:
            best_individual = min(individual_avgs)
            improvement = (best_individual - stacking_avg) / best_individual * 100
            print(f"\n🎯 스태킹 개선율: {improvement:.1f}%")

print(f"\n📁 출력 파일:")
print(f"   📊 상세 결과: advanced_stacking_ensemble_predictions.csv")
print(f"   🏆 제출용: submission_advanced_stacking.csv")

print(f"\n🎉 지역난방 열수요 예측 완료!")
print(f"🚀 세계 최고 수준의 고도화된 스태킹 앙상블 적용 성공!")

# GPU 메모리 정리
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"🧹 GPU 메모리 정리 완료")

print(f"\n💡 재현성 보장: 동일한 시드({SEED})로 언제든 동일한 결과 재현 가능")
print(f"🎯 Data Leakage 방지를 위해 추가 수정이 필요할 수 있습니다.")